# 03. Preprocesamiento y filtrado de las trayectorias

Prepara el panel para el clustering y la segmentación: validación del muestreo mensual, limpieza de picos, interpolación de faltantes, transformación, estacionaridad, suavizado (media móvil y pasa-bajos FFT) y crecimiento mensual.

**Entrada:** panel long del notebook 02.
**Salida:** panel preprocesado, con la serie cruda y las marcas de cada etapa.

In [ ]:
# PAQUETES

library(data.table)
library(tseries)    # adf.test y kpss.test
library(grid)
library(gridExtra)  # tablas exportables como imagen
library(ggplot2)
library(ggrepel)    # etiquetas que no se superponen

# Tipografias de la presentacion (se dibujan con ragg): Berlin Sans FB Demi
# para titulos y DejaVu Sans Mono para el resto (copia en fuentes/)
systemfonts::register_font("LWR Titulo", plain = "C:/Windows/Fonts/BRLNSDB.TTF")
systemfonts::register_font(
  "LWR Mono",
  plain = "fuentes/DejaVuSansMono.ttf",
  bold = "fuentes/DejaVuSansMono-Bold.ttf"
)

## 1. Configuración y carga

In [ ]:
# ============================================================
# CONFIGURACION
# ============================================================

# Panel long exportado por el notebook 02: el más reciente del
# directorio (el nombre cambia con la cantidad de artistas)
archivos_panel_long_disponibles <- list.files(
  pattern = "^artistas_ST_[0-9]+de[0-9]+_top[0-9]+_22_26_na_long[.]csv$"
)

stopifnot(
  "No hay panel long exportado: correr el notebook 02" =
    length(archivos_panel_long_disponibles) > 0
)

ARCHIVO_PANEL_LONG <- archivos_panel_long_disponibles[
  which.max(file.mtime(archivos_panel_long_disponibles))
]

FIGURES_DIR <- "figuras"
dir.create(FIGURES_DIR, showWarnings = FALSE, recursive = TRUE)

VARIABLES_PANEL <- c(
  spotify_followers           = "Seguidores en Spotify",
  instagram_followers         = "Seguidores en Instagram",
  spotify_playlist_count      = "Playlists en Spotify",
  youtube_channel_subscribers = "Suscriptores en YouTube",
  youtube_playlist_count      = "Playlists en YouTube"
)

# Variables de audiencia acumulada: no pueden caer a la mitad de
# un mes a otro, por eso solo en ellas se buscan picos aislados y
# cambios de nivel (bloque 4). Las cuentas de playlists suben y bajan.
VARIABLES_STOCK_AUDIENCIA <- c(
  "spotify_followers",
  "instagram_followers",
  "youtube_channel_subscribers"
)

# Salto mensual (en log1p) que define un error de la fuente:
# log(2) es duplicarse o caer a la mitad de un mes a otro
UMBRAL_SALTO_FUENTE <- log(2)

# Quiebre comun de la fuente (bloque 4): mes en que al menos 2/3
# de los artistas cambian mas de 10 % en el mismo sentido
PROPORCION_QUIEBRE_COMUN <- 2 / 3
CAMBIO_MINIMO_QUIEBRE_COMUN <- log(1.1)

# Transformacion de las series (bloque 5). Opciones:
# "log1p_centrado" (elegida), "log1p_estandarizado",
# "log1p_desde_inicio", "sin_transformar"
TRANSFORMACION <- "log1p_centrado"

# Medias moviles centradas, en meses (bloque 7): suavizado con
# ventana impar simple; tendencia con media movil 2x12 (13 meses,
# extremos a mitad de peso), el filtro de decompose() mensual
VENTANAS_MEDIA_MOVIL <- c(
  suavizado = 3,
  tendencia = 12
)

# Armonicos que conserva el pasa-bajos FFT, ademas de la
# frecuencia cero (bloque 8)
BANDA_FILTRO_PASA_BAJOS <- 6

# Artistas ilustrativos del suavizado (bloque 7): uno por tercil
# de cm_artist_rank, de reconocimiento publico y sin meses
# imputados en ninguna variable
ARTISTAS_ILUSTRATIVOS <- c(
  rank_alto  = 206907L,   # TINI
  rank_medio = 276L,      # Megadeth
  rank_bajo  = 2456L      # Abel Pintos
)

# Artistas excluidos por anomalías de la fuente revisadas a mano
# (bloque 4). Las series constantes se detectan solas.
ARTISTAS_ANOMALIA_FUENTE <- c(
  729342L     # Bhad Bhabie: suscriptores de YouTube entre 5 y 7 en los 49 meses
)

# Features de crecimiento (bloque 10): el maximo y la caida se buscan
# sin los 2 primeros ni los 2 ultimos meses. En el borde no hay
# pasado (o futuro) observado y la derivada y la media movil usan un
# solo lado: el inicio se lee como basal, no como explosion
MESES_BORDE_SIN_EXTREMOS <- 2L

# Variable para los ejemplos de una sola serie (bloques 7 y 8)
VARIABLE_ILUSTRATIVA <- "spotify_followers"

# Artistas fijos de la figura de presentacion: trayectoria
# estable y crecimiento explosivo en 2023, ambos del decil 1
ARTISTAS_FIGURA_PRESENTACION <- c(
  3648L,      # Ed Sheeran
  4251908L    # Peso Pluma
)

# Series diarias del piloto de extraccion, para verificar el
# muestreo mensual (bloque 3), y su correspondencia con el panel
ARCHIVO_DIARIO_PILOTO <- "chartmetric_pilotos/chartmetric_raw_long.rds"

FUENTE_DIARIA_VARIABLES <- data.table(
  source   = c("spotify", "instagram", "youtube_channel"),
  metric   = c("followers", "followers", "subscribers"),
  variable = VARIABLES_STOCK_AUDIENCIA
)

ARCHIVO_PANEL_PREPROCESADO <- sub(
  "_na_long\\.csv$",
  paste0("_preprocesado_", TRANSFORMACION, "_long.csv"),
  ARCHIVO_PANEL_LONG
)

# Estilo grafico comun. Cada color tiene un solo significado en
# todo el notebook:
#   PALETA_ARTISTAS: un color por artista (figuras 1 y 4)
#   COLOR_SERIE_BASE: dato de partida (serie cruda, transformada o diaria)
#   COLOR_RESULTADO: serie que queda despues de corregir, imputar o filtrar
#   COLOR_ALERTA: problema detectado o alternativa descartada
#   COLOR_MEDIA_MOVIL y COLOR_TENDENCIA: los dos filtros de media movil
#   COLOR_REFERENCIA: lineas de referencia (Nyquist, corte en 0,7)
COLOR_FONDO <- "#FFFFFF"
COLOR_GRILLA <- "#EDEDED"
PALETA_ARTISTAS <- c("#111111", "#B3B3B3", "#5E5E5E")
COLOR_SERIE_BASE <- "#BDBDBD"
COLOR_RESULTADO <- "#111111"
COLOR_ALERTA <- "#C8102E"   # único acento: anomalías y lo que se pierde
COLOR_MEDIA_MOVIL <- "#111111"
COLOR_TENDENCIA <- "#8C8C8C"
COLOR_REFERENCIA <- "#6B6B6B"

ARCHIVO_PANEL_PREPROCESADO

In [ ]:
# ============================================================
# CARGA DEL PANEL EN FORMATO LARGO
# Una sola tabla, panel_long, que los bloques siguientes
# completan por referencia con una columna por etapa
# ============================================================

panel_long <- fread(
  ARCHIVO_PANEL_LONG,
  select = c(
    "artist_id", "artist_name", "cm_artist_rank", "decil_marco",
    "variable", "month", "value", "tipo_dato", "interpolated"
  )
)

setnames(panel_long, "value", "valor_crudo")
panel_long[, month := as.Date(month)]
setkey(panel_long, artist_id, variable, month)

MESES_PANEL <- sort(unique(panel_long$month))
N_MESES_PANEL <- length(MESES_PANEL)
panel_long[, mes_indice := match(month, MESES_PANEL)]

artistas_panel <- unique(
  panel_long[, .(artist_id, artist_name, cm_artist_rank, decil_marco)]
)

cat(
  "Artistas:", nrow(artistas_panel), "\n",
  "Variables:", uniqueN(panel_long$variable), "\n",
  "Meses:", N_MESES_PANEL, "(", format(MESES_PANEL[1]), "a", format(MESES_PANEL[N_MESES_PANEL]), ")\n"
)

In [ ]:
# ------------------------------------------------------------
# CONTROL: grilla completa (49 meses por artista y variable) y
# celdas sin dato por variable segun su ubicacion en la serie
# ------------------------------------------------------------

stopifnot(
  panel_long[, .N, by = .(artist_id, variable)][, all(N == N_MESES_PANEL)],
  setequal(unique(panel_long$variable), names(VARIABLES_PANEL))
)

dcast(
  panel_long[, .N, by = .(variable, tipo_dato)][, pct := round(100 * N / sum(N), 2), by = variable],
  variable ~ tipo_dato,
  value.var = "pct",
  fill = 0
)

## 2. Las series crudas

Las variables difieren en órdenes de magnitud entre sí y entre artistas: hace falta transformarlas antes de comparar formas. Ejemplos: Ed Sheeran (trayectoria consolidada) y Peso Pluma (crecimiento explosivo en 2023).

In [ ]:
# ============================================================
# FUNCIONES GRAFICAS COMUNES
# ============================================================

MARCAS_X_FIGURA <- seq(MESES_PANEL[1], MESES_PANEL[N_MESES_PANEL], by = "year")

# Muestra el grafico en el notebook y lo guarda en FIGURES_DIR
guardar_figura <- function(nombre_archivo, funcion_grafico, ancho, alto) {
  # se dibuja con ragg (conoce las tipografías registradas) en un archivo
  # temporal y se escriben sus bytes sobre el destino: evita fallas si el
  # PNG anterior está abierto en un visor (file.copy no puede reemplazarlo)
  archivo_temporal <- tempfile(fileext = ".png")
  ragg::agg_png(archivo_temporal, width = ancho, height = alto, res = 200)
  par(family = "LWR Mono")
  funcion_grafico()
  invisible(dev.off())
  bytes_figura <- readBin(archivo_temporal, "raw", file.size(archivo_temporal))
  # hasta cinco intentos: un visor que relee el archivo lo bloquea
  # por un instante
  guardado <- FALSE
  for (intento in 1:5) {
    guardado <- tryCatch({
      writeBin(bytes_figura, file.path(FIGURES_DIR, nombre_archivo))
      TRUE
    }, error = function(e) FALSE, warning = function(w) FALSE)
    if (guardado) break
    Sys.sleep(1)
  }
  if (!guardado) warning("No se pudo guardar ", nombre_archivo, ": el archivo puede estar abierto")
  # el dispositivo del notebook no conoce las tipografías: se muestra el PNG
  IRdisplay::display_png(file = archivo_temporal)
}

# Etiquetas del eje y en millones o miles, sin notacion cientifica
etiqueta_eje_magnitud <- function(x) {
  numero <- function(v) formatC(v, format = "fg", digits = 3, big.mark = ".", decimal.mark = ",")
  ifelse(x >= 1e6, paste0(numero(x / 1e6), " M"),
         ifelse(x >= 1e3, paste0(numero(x / 1e3), " mil"), numero(x)))
}

# Nombre de variable dentro de una frase: solo la inicial en minuscula
en_minuscula_inicial <- function(texto) paste0(tolower(substr(texto, 1, 1)), substring(texto, 2))

# Lienzo vacio con eje temporal del panel y grilla horizontal
abrir_lienzo <- function(rango_y, titulo, escala_log = FALSE, tamanio_titulo = 1.1) {
  plot(
    range(MESES_PANEL), rango_y,
    type = "n", log = if (escala_log) "y" else "",
    xlab = "", ylab = "", xaxt = "n", yaxt = "n", bty = "n",
    main = titulo, font.main = 1, cex.main = tamanio_titulo
  )
  axis(1, at = MARCAS_X_FIGURA, labels = paste("mar", format(MARCAS_X_FIGURA, "%Y")),
       col = NA, col.ticks = "#BDBDBD")
  marcas_y <- axTicks(2)
  axis(2, at = marcas_y, col = NA, col.ticks = NA,
       labels = if (max(abs(marcas_y)) >= 1e3) etiqueta_eje_magnitud(marcas_y) else format(marcas_y, decimal.mark = ","))
  abline(h = marcas_y, col = COLOR_GRILLA)
}

# Las cinco variables, un panel por variable, para los artistas
# de la figura de presentacion; columna_valor elige la etapa
grafico_cinco_variables <- function(columna_valor, titulo, escala_log) {

  series_figura <- panel_long[artist_id %in% ARTISTAS_FIGURA_PRESENTACION]

  # con 5 paneles en fila R reduce los textos a 2/3: el titulo de
  # cada panel se agranda para compensar
  par(mfrow = c(1, 5), bg = COLOR_FONDO, mar = c(3, 4.2, 3.6, 1),
      oma = c(0, 0, 6, 0), las = 1, cex.axis = 0.9)

  for (variable_figura in names(VARIABLES_PANEL)) {

    series_variable <- series_figura[variable == variable_figura]

    abrir_lienzo(range(series_variable[[columna_valor]], na.rm = TRUE),
                 VARIABLES_PANEL[[variable_figura]], escala_log, tamanio_titulo = 1.4)

    for (i in seq_along(ARTISTAS_FIGURA_PRESENTACION)) {
      serie_artista <- series_variable[artist_id == ARTISTAS_FIGURA_PRESENTACION[i]]
      # meses imputados (si ya existe la marca) en linea punteada
      # fina, observados en linea llena
      valores_observados <- serie_artista[[columna_valor]]
      if ("imputado" %in% names(serie_artista)) valores_observados[serie_artista$imputado] <- NA
      lines(serie_artista$month, serie_artista[[columna_valor]], col = PALETA_ARTISTAS[i], lwd = 1.5, lty = 3)
      lines(serie_artista$month, valores_observados, col = PALETA_ARTISTAS[i], lwd = 2.5)
    }
  }

  mtext(titulo, side = 3, outer = TRUE, line = 3.4, cex = 1.9, family = "LWR Titulo")

  par(fig = c(0, 1, 0, 1), oma = c(0, 0, 0, 0), mar = c(0, 0, 0, 0), new = TRUE)
  plot(0, 0, type = "n", bty = "n", xaxt = "n", yaxt = "n")
  legend(
    "top",
    legend = artistas_panel[match(ARTISTAS_FIGURA_PRESENTACION, artist_id), artist_name],
    col = PALETA_ARTISTAS[seq_along(ARTISTAS_FIGURA_PRESENTACION)],
    lwd = 2.5, horiz = TRUE, bty = "n", inset = c(0, 0.075), cex = 1.05
  )
}

In [ ]:
# ============================================================
# FIGURA 1: series crudas de los dos artistas de presentacion.
# Dos versiones: eje logarítmico (se ven las dos trayectorias) y eje
# lineal (el artista chico queda aplastado: el problema de magnitudes)
# ============================================================

TITULO_SERIES_CRUDAS <- "El problema de las magnitudes en las series crudas"

guardar_figura(
  "prepro_01_series_crudas.png",
  function() grafico_cinco_variables("valor_crudo", TITULO_SERIES_CRUDAS, escala_log = TRUE),
  ancho = 3000, alto = 950
)

guardar_figura(
  "prepro_01_series_crudas_lineal.png",
  function() grafico_cinco_variables("valor_crudo", TITULO_SERIES_CRUDAS, escala_log = FALSE),
  ancho = 3000, alto = 950
)


**Estadística descriptiva por mes:** media, mediana y percentiles entre artistas, sobre los valores crudos.

In [ ]:
# ============================================================
# ESTADISTICOS DESCRIPTIVOS POR VARIABLE Y MES: media, mediana,
# desvio y percentiles entre artistas, valores crudos
# ============================================================

descriptivos_mensuales <- panel_long[
  !is.na(valor_crudo),
  .(artistas = .N,
    media = mean(valor_crudo),
    mediana = median(valor_crudo),
    desvio = sd(valor_crudo),
    p10 = quantile(valor_crudo, 0.10),
    p25 = quantile(valor_crudo, 0.25),
    p75 = quantile(valor_crudo, 0.75),
    p90 = quantile(valor_crudo, 0.90)),
  keyby = .(variable, month)
]

# Control: un registro por variable y mes, y vista del primer y
# el ultimo mes
stopifnot(descriptivos_mensuales[, .N, by = variable][, all(N == N_MESES_PANEL)])

descriptivos_mensuales[month %in% range(MESES_PANEL), .(variable, month, artistas, media = round(media), mediana = round(mediana))]


In [ ]:
# ============================================================
# TABLA 1: trayectoria mediana de cada variable, inicio y final
# del panel (imagen para presentaciones) y CSV con todos los meses
# ============================================================

MES_INICIO_PANEL <- MESES_PANEL[1]
MES_FINAL_PANEL <- MESES_PANEL[N_MESES_PANEL]
N_ARTISTAS_DESCRIPTIVA <- uniqueN(panel_long$artist_id)

formato_porcentaje <- function(x) paste0(ifelse(x > 0, "+", ""), trimws(format(round(x), big.mark = ".", decimal.mark = ",")), " %")
formato_rango <- function(desde, hasta) paste(trimws(etiqueta_eje_magnitud(desde)), "a", trimws(etiqueta_eje_magnitud(hasta)))

inicio_final <- merge(
  descriptivos_mensuales[month == MES_INICIO_PANEL],
  descriptivos_mensuales[month == MES_FINAL_PANEL],
  by = "variable", suffixes = c("_inicio", "_final")
)[match(names(VARIABLES_PANEL), variable)]

tabla_1_descriptiva <- inicio_final[
  ,
  .(`Variable` = unname(VARIABLES_PANEL[variable]),
    `Mediana inicial` = trimws(etiqueta_eje_magnitud(mediana_inicio)),
    `Mediana final` = trimws(etiqueta_eje_magnitud(mediana_final)),
    `Cambio de la mediana` = formato_porcentaje(100 * (mediana_final / mediana_inicio - 1)),
    `Mitad central inicial` = formato_rango(p25_inicio, p75_inicio),
    `Mitad central final` = formato_rango(p25_final, p75_final),
    `Media / mediana final` = format(round(media_final / mediana_final, 1), decimal.mark = ","))
]

titulo_tabla_1 <- paste0(
  "Tabla 1. Trayectoria mediana de cada variable, ", format(MES_INICIO_PANEL, "%B %Y"),
  " a ", format(MES_FINAL_PANEL, "%B %Y"), " (", N_ARTISTAS_DESCRIPTIVA, " artistas)"
)

nota_tabla_1 <- paste(
  "Mediana y percentiles entre artistas, calculados mes a mes sobre los valores crudos, antes de limpieza y transformación.",
  "Mitad central: del percentil 25 al 75, donde está la mitad de los artistas. M: millones; mil: miles.",
  "Media / mediana mayor que 1: pocos artistas con valores muy altos elevan la media por encima del artista típico.",
  "Playlists en YouTube: la mayor parte del cambio son dos saltos de la fuente en 2022, que se corrigen en la sección 4.",
  sep = "\n"
)

tema_tabla_1 <- ttheme_minimal(
  base_size = 13,
  core = list(
    bg_params = list(fill = c(COLOR_FONDO, "#F2F2F2"), col = NA),
    fg_params = list(hjust = 1, x = 0.95)
  ),
  colhead = list(
    fg_params = list(fontface = "bold", hjust = 1, x = 0.95),
    bg_params = list(fill = COLOR_FONDO, col = NA)
  )
)

grob_tabla_1 <- tableGrob(tabla_1_descriptiva, rows = NULL, theme = tema_tabla_1)

# primera columna (nombre de la variable) alineada a la izquierda
for (indice in which(grob_tabla_1$layout$l == 1 & grob_tabla_1$layout$name %in% c("core-fg", "colhead-fg"))) {
  grob_tabla_1$grobs[[indice]]$hjust <- 0
  grob_tabla_1$grobs[[indice]]$x <- unit(0.05, "npc")
}

# linea bajo el encabezado
grob_tabla_1 <- gtable::gtable_add_grob(
  grob_tabla_1,
  segmentsGrob(x0 = unit(0, "npc"), x1 = unit(1, "npc"), y0 = unit(0, "npc"), y1 = unit(0, "npc"),
               gp = gpar(col = COLOR_RESULTADO, lwd = 1.5)),
  t = 1, l = 1, r = ncol(grob_tabla_1)
)

grob_tabla_1_completa <- arrangeGrob(
  grob_tabla_1,
  top = textGrob(titulo_tabla_1, x = 0.02, hjust = 0, gp = gpar(fontsize = 16, fontface = "bold")),
  bottom = textGrob(nota_tabla_1, x = 0.02, hjust = 0, gp = gpar(fontsize = 11, col = COLOR_REFERENCIA))
)

guardar_figura(
  "tabla_01_descriptiva.png",
  function() {
    grid.newpage()
    grid.rect(gp = gpar(fill = COLOR_FONDO, col = NA))
    grid.draw(grob_tabla_1_completa)
  },
  ancho = 3000, alto = 740
)

# CSV con todos los meses y estadisticos, sin redondear
fwrite(
  descriptivos_mensuales[, .(variable, nombre = unname(VARIABLES_PANEL[variable]), month, artistas,
                             media, mediana, desvio_estandar = desvio, p10, p25, p75, p90)],
  file.path(FIGURES_DIR, "tabla_01_descriptiva.csv")
)


In [ ]:
# ============================================================
# FIGURA 1b: trayectoria mediana de cada variable entre artistas,
# con la mitad central (P25 a P75), el 80 % central (P10 a P90)
# y la media
# ============================================================

TAMANIO_TITULO_DESCRIPTIVA <- 2.4
TAMANIO_TITULO_PANEL_DESCRIPTIVA <- 1.5
TAMANIO_TEXTO_DESCRIPTIVA <- 1.1

COLOR_BANDA_P10_P90 <- "#E4E4E4"
COLOR_BANDA_P25_P75 <- "#C4C4C4"

banda_entre <- function(meses, desde, hasta, color) {
  polygon(c(meses, rev(meses)), c(desde, rev(hasta)), col = color, border = NA)
}

grafico_trayectoria_mediana <- function() {

  par(mfrow = c(1, length(VARIABLES_PANEL)), bg = COLOR_FONDO, mar = c(3, 4.5, 3, 1),
      oma = c(0, 0, 7.5, 0), las = 1, cex.axis = TAMANIO_TEXTO_DESCRIPTIVA)

  for (variable_figura in names(VARIABLES_PANEL)) {

    estadisticos_variable <- descriptivos_mensuales[variable_figura]

    abrir_lienzo(range(estadisticos_variable[, .(p10, p90, media)]), "", escala_log = TRUE)
    title(VARIABLES_PANEL[[variable_figura]], font.main = 1, cex.main = TAMANIO_TITULO_PANEL_DESCRIPTIVA, line = 1.2)

    banda_entre(estadisticos_variable$month, estadisticos_variable$p10, estadisticos_variable$p90, COLOR_BANDA_P10_P90)
    banda_entre(estadisticos_variable$month, estadisticos_variable$p25, estadisticos_variable$p75, COLOR_BANDA_P25_P75)
    lines(estadisticos_variable$month, estadisticos_variable$media, col = COLOR_REFERENCIA, lwd = 2, lty = 2)
    lines(estadisticos_variable$month, estadisticos_variable$mediana, col = COLOR_RESULTADO, lwd = 3)
    points(estadisticos_variable$month, estadisticos_variable$mediana, pch = 19, col = COLOR_RESULTADO, cex = 0.7)
  }

  mtext("Trayectoria mediana de cada variable entre artistas (escala logarítmica)",
        side = 3, outer = TRUE, line = 4.6, font = 2, cex = TAMANIO_TITULO_DESCRIPTIVA, family = "LWR Titulo")

  par(fig = c(0, 1, 0, 1), oma = c(0, 0, 0, 0), mar = c(0, 0, 0, 0), new = TRUE)
  plot(0, 0, type = "n", bty = "n", xaxt = "n", yaxt = "n")
  par(lend = 1)  # extremos rectos: las muestras de las bandas no invaden el texto
  legend("top", ncol = 4, bty = "n", inset = c(0, 0.11), cex = TAMANIO_TEXTO_DESCRIPTIVA, x.intersp = 1,
         legend = c("mediana (artista típico)", "media", "mitad central (P25 a P75)", "80 % central (P10 a P90)"),
         col = c(COLOR_RESULTADO, COLOR_REFERENCIA, COLOR_BANDA_P25_P75, COLOR_BANDA_P10_P90),
         lwd = c(3, 2, 10, 10), lty = c(1, 2, 1, 1), pch = c(19, NA, NA, NA), seg.len = 2)
}

guardar_figura("prepro_01b_descriptiva.png", grafico_trayectoria_mediana, ancho = 3200, alto = 1150)


La mediana crece en las cinco variables y las bandas avanzan paralelas en escala log. La media supera siempre a la mediana por la asimetría de los artistas masivos: de ahí la escala logarítmica. Los saltos de 2022 en playlists de YouTube son cambios de la fuente (bloque 4).

## 3. Muestreo mensual

El panel toma el último dato diario de cada mes (Nyquist: 6 ciclos por año). Se mide la energía de las series diarias del piloto por encima de ese límite.

In [ ]:
# ============================================================
# SERIES DIARIAS: MUESTREO UNIFORME Y ESPECTRO
# ============================================================

DIAS_PANEL <- seq(MESES_PANEL[1], seq(MESES_PANEL[N_MESES_PANEL], by = "month", length.out = 2)[2] - 1, by = "day")
N_DIAS_PANEL <- length(DIAS_PANEL)
FRECUENCIA_NYQUIST_MENSUAL <- 12 / 2    # ciclos por anio

serie_diaria <- readRDS(ARCHIVO_DIARIO_PILOTO)[
  FUENTE_DIARIA_VARIABLES, on = .(source, metric), nomatch = NULL
][
  , `:=`(artist_id = as.integer(artist_id), fecha = as.Date(timestamp))
][
  artist_id %in% artistas_panel$artist_id & fecha %between% range(DIAS_PANEL) & !is.na(value),
  .(valor = value[.N]),
  keyby = .(artist_id, variable, fecha)
]

serie_diaria_uniforme <- serie_diaria[
  ,
  .(fecha = DIAS_PANEL,
    log_valor = approx(as.numeric(fecha), log1p(valor), xout = as.numeric(DIAS_PANEL), rule = 2)$y),
  by = .(artist_id, variable)
]

armonicos_diarios <- seq_len(N_DIAS_PANEL %/% 2)

espectro_diario <- serie_diaria_uniforme[
  ,
  .(ciclos_por_anio = armonicos_diarios * 365.25 / N_DIAS_PANEL,
    potencia = Mod(fft(residuals(lm(log_valor ~ seq_along(log_valor)))))[armonicos_diarios + 1]^2),
  by = .(artist_id, variable)
]

# Dos versiones mensuales de la misma serie diaria: el ultimo dia
# (la del panel) y el promedio del mes (pasa-bajos antes de submuestrear)
muestreo_mensual_comparado <- serie_diaria_uniforme[
  ,
  .(ultimo_dia = log_valor[.N], promedio_mes = mean(log_valor)),
  by = .(artist_id, variable, month = as.Date(format(fecha, "%Y-%m-01")))
]

resumen_muestreo <- espectro_diario[
  ,
  .(pct_energia = 100 * sum(potencia[ciclos_por_anio > FRECUENCIA_NYQUIST_MENSUAL]) / sum(potencia)),
  by = .(artist_id, variable)
][
  ,
  .(series = .N,
    pct_energia_sobre_nyquist_mediana = round(median(pct_energia), 2),
    pct_energia_sobre_nyquist_p90 = round(quantile(pct_energia, 0.9), 2)),
  by = variable
][
  muestreo_mensual_comparado[
    , .(dif_ultimo_dia_vs_promedio_pct = round(100 * median(abs(ultimo_dia - promedio_mes)), 2)), by = variable
  ],
  on = "variable"
]

resumen_muestreo

In [ ]:
# ============================================================
# FIGURA 2: espectro diario y un alias por muestreo mensual
# ============================================================

espectro_diario_mediano <- espectro_diario[
  , .(potencia_mediana = median(potencia)), by = ciclos_por_anio
]

# Caso de alias: el mes cuyo ultimo dia mas se aparta del promedio
# del mes (un error de pocos dias que cae sobre la fecha muestreada)
caso_alias <- muestreo_mensual_comparado[which.max(abs(ultimo_dia - promedio_mes))]

serie_diaria_alias <- serie_diaria_uniforme[
  artist_id == caso_alias$artist_id & variable == caso_alias$variable &
    fecha %between% (caso_alias$month + c(-75, 105))
]

grafico_muestreo <- function() {

  par(mfrow = c(1, 2), bg = COLOR_FONDO, mar = c(4.5, 4.5, 4, 1), las = 1, cex.axis = 0.85)

  # a) espectro diario mediano y limite de Nyquist mensual
  plot(espectro_diario_mediano$ciclos_por_anio, espectro_diario_mediano$potencia_mediana,
       type = "n", log = "xy", bty = "n", xlab = "Ciclos por año", ylab = "Potencia mediana", yaxt = "n",
       main = "Espectro de las series diarias (sin tendencia lineal)", font.main = 1, cex.main = 1.1)
  rect(FRECUENCIA_NYQUIST_MENSUAL, 10^par("usr")[3], 10^par("usr")[2], 10^par("usr")[4], col = "#F2F2F2", border = NA)
  lines(espectro_diario_mediano$ciclos_por_anio, espectro_diario_mediano$potencia_mediana, col = COLOR_SERIE_BASE, lwd = 2.5)
  abline(v = FRECUENCIA_NYQUIST_MENSUAL, col = COLOR_REFERENCIA, lwd = 2, lty = 2)
  text(FRECUENCIA_NYQUIST_MENSUAL * 1.15, 10^par("usr")[4], adj = c(0, 1.5), cex = 0.85,
       "Nyquist del muestreo mensual\n(6 ciclos por año)")

  # b) serie diaria con un error de pocos dias y sus muestras mensuales
  meses_zoom <- muestreo_mensual_comparado[
    artist_id == caso_alias$artist_id & variable == caso_alias$variable &
      month %between% range(serie_diaria_alias$fecha)
  ]
  fechas_ultimo_dia <- seq(meses_zoom$month[1], by = "month", length.out = nrow(meses_zoom) + 1)[-1] - 1
  plot(serie_diaria_alias$fecha, expm1(serie_diaria_alias$log_valor), type = "n", log = "y",
       bty = "n", xlab = "", ylab = "", yaxt = "n",
       main = paste0(artistas_panel[artist_id == caso_alias$artist_id, artist_name], ", ",
                     en_minuscula_inicial(VARIABLES_PANEL[[caso_alias$variable]]), ": serie diaria y muestra mensual"),
       font.main = 1, cex.main = 1.1)
  marcas_y <- axTicks(2)
  axis(2, at = marcas_y, labels = etiqueta_eje_magnitud(marcas_y), col = NA)
  abline(h = marcas_y, col = COLOR_GRILLA)
  lines(serie_diaria_alias$fecha, expm1(serie_diaria_alias$log_valor), col = COLOR_SERIE_BASE, lwd = 1.5)
  points(fechas_ultimo_dia, expm1(meses_zoom$ultimo_dia), pch = 19, col = COLOR_ALERTA, cex = 1.4)
  points(fechas_ultimo_dia, expm1(meses_zoom$promedio_mes), pch = 21, col = COLOR_RESULTADO, lwd = 2, cex = 1.4)
  legend("bottomleft", bty = "n", cex = 0.85, lwd = c(1.5, NA, NA), pch = c(NA, 19, 1),
         col = c(COLOR_SERIE_BASE, COLOR_ALERTA, COLOR_RESULTADO),
         legend = c("serie diaria", "último día del mes (panel)", "promedio del mes"))
}

guardar_figura("prepro_02_muestreo_mensual.png", grafico_muestreo, ancho = 2600, alto = 1000)

La energía sobre Nyquist es del orden del 1% y el último día difiere del promedio mensual en menos del 1%: el muestreo mensual representa bien las audiencias. Los errores breves de la fuente pueden convertirse en picos aislados (bloque 4).

Versión para el video: el mismo espectro con la energía de cada lado del límite de Nyquist rotulada.

In [ ]:
# ============================================================
# FIGURA 2b (video): energia por debajo y por encima de Nyquist
# ============================================================

COLOR_ENERGIA_LENTA <- COLOR_RESULTADO
COLOR_ENERGIA_RAPIDA <- COLOR_ALERTA

# Porcentaje de energia sobre Nyquist por serie (mismo calculo que
# resumen_muestreo) y su mediana entre todas las series
energia_rapida_por_serie <- espectro_diario[
  , .(pct_rapida = 100 * sum(potencia[ciclos_por_anio > FRECUENCIA_NYQUIST_MENSUAL]) / sum(potencia)),
  by = .(artist_id, variable)
]
pct_rapida_global <- energia_rapida_por_serie[, median(pct_rapida)]

# Sanity check: la mediana global queda entre las medianas por variable
stopifnot(pct_rapida_global >= min(resumen_muestreo$pct_energia_sobre_nyquist_mediana) - 0.01,
          pct_rapida_global <= max(resumen_muestreo$pct_energia_sobre_nyquist_mediana) + 0.01)

formato_pct <- function(x) paste0(format(round(x, 1), nsmall = 1, decimal.mark = ","), "%")

grafico_muestreo_video <- function() {

  par(bg = COLOR_FONDO, mar = c(6, 2, 7, 2), las = 1, cex.axis = 1.1)

  plot(espectro_diario_mediano$ciclos_por_anio, espectro_diario_mediano$potencia_mediana,
       type = "n", log = "xy", bty = "n", xlab = "", ylab = "", xaxt = "n", yaxt = "n")
  marcas_x <- c(0.2, 0.5, 1, 2, 5, 10, 20, 50, 100, 200)
  axis(1, at = marcas_x, labels = format(marcas_x, decimal.mark = ",", drop0trailing = TRUE, trim = TRUE))
  title(xlab = "Ciclos por año", line = 3.2, cex.lab = 1.2)
  limites <- 10^par("usr")
  rect(FRECUENCIA_NYQUIST_MENSUAL, limites[3], limites[2], limites[4],
       col = adjustcolor(COLOR_ENERGIA_RAPIDA, alpha.f = 0.1), border = NA)
  lines(espectro_diario_mediano$ciclos_por_anio, espectro_diario_mediano$potencia_mediana,
        col = COLOR_SERIE_BASE, lwd = 3)
  abline(v = FRECUENCIA_NYQUIST_MENSUAL, col = COLOR_REFERENCIA, lwd = 2, lty = 2)

  # cada cifra en la zona libre de su lado: la curva baja de izquierda a derecha
  centro_lento <- sqrt(limites[1] * FRECUENCIA_NYQUIST_MENSUAL)
  centro_rapido <- sqrt(FRECUENCIA_NYQUIST_MENSUAL * limites[2])
  altura_relativa <- function(fraccion) limites[3] * (limites[4] / limites[3])^fraccion
  text(centro_lento, altura_relativa(0.25), formato_pct(100 - pct_rapida_global), cex = 3.6, font = 2, col = COLOR_ENERGIA_LENTA)
  text(centro_rapido, altura_relativa(0.78), formato_pct(pct_rapida_global), cex = 3.6, font = 2, col = COLOR_ENERGIA_RAPIDA)
  mtext(c("oscilaciones de más de 2 meses\nun dato por mes las ve",
          "oscilaciones de menos de 2 meses\nun dato por mes no las ve"),
        side = 3, line = 0.6, at = c(centro_lento, centro_rapido), cex = 1.05,
        col = c(COLOR_ENERGIA_LENTA, COLOR_ENERGIA_RAPIDA))
  text(FRECUENCIA_NYQUIST_MENSUAL, limites[3], "límite de Nyquist\n(6 ciclos por año)",
       adj = c(-0.05, -0.3), cex = 0.95, col = COLOR_REFERENCIA)
  mtext("¿Qué se pierde con un dato por mes?", side = 3, line = 4.2, at = limites[1], adj = 0,
        cex = 2.2, col = COLOR_RESULTADO, family = "LWR Titulo")
  mtext(paste("Energía de", nrow(energia_rapida_por_serie),
              "series diarias del piloto (Spotify, Instagram y YouTube), sin tendencia lineal; mediana por serie"),
        side = 1, line = 4.8, at = limites[1], adj = 0, cex = 0.9, col = COLOR_REFERENCIA)
}

guardar_figura("prepro_02b_muestreo_mensual_video.png", grafico_muestreo_video, ancho = 2400, alto = 1400)

**Muestreo en amplitud:** los suscriptores de YouTube vienen redondeados a tres cifras significativas. Se mide la proporción de valores redondeados y el escalón relativo frente al cambio mensual típico.

In [ ]:
# ============================================================
# MUESTREO EN AMPLITUD: redondeo de los conteos en la fuente
# ============================================================

cifras_significativas <- function(x) {
  nchar(sub("0+$", "", format(round(x), scientific = FALSE, trim = TRUE)))
}

# Proporcion de valores (desde 1000) con tres cifras significativas
# o menos: en un conteo exacto es rara, en uno redondeado es la norma
panel_long[
  !is.na(valor_crudo) & valor_crudo >= 1e3,
  .(pct_valores_hasta_3_cifras = round(100 * mean(cifras_significativas(valor_crudo) <= 3), 1)),
  by = variable
]

# Escalon de redondeo relativo al valor (Delta x / x) frente al cambio
# mensual tipico de los suscriptores de YouTube, ambos en %
suscriptores_youtube <- panel_long[variable == "youtube_channel_subscribers" & !is.na(valor_crudo)]

cat(
  "Escalón de redondeo mediano:",
  round(100 * suscriptores_youtube[valor_crudo >= 1e3, median(10^(floor(log10(valor_crudo)) - 2) / valor_crudo)], 2), "%
",
  "Cambio mensual mediano:",
  round(100 * suscriptores_youtube[, .(cambio = abs(diff(log1p(valor_crudo)))), by = artist_id][, median(cambio, na.rm = TRUE)], 2), "%
"
)

# Ejemplo: primeros meses de Ed Sheeran
panel_long[.(ARTISTAS_FIGURA_PRESENTACION[1], "youtube_channel_subscribers"), head(valor_crudo, 6)]

Seis de cada diez valores de suscriptores de YouTube están redondeados. El escalón es del orden de la mitad del cambio mensual típico: genera escalones que la media móvil de 3 meses reparte.

## 4. Picos aislados y faltantes

Sobre `log1p` del valor:

- **Picos aislados** (sólo audiencias acumuladas): duplicarse o caer a la mitad y volver al mes siguiente se trata como faltante.
- **Cambios de nivel:** caída a menos de la mitad que no se revierte. El artista se excluye.
- **Series sin variación y anomalías de la fuente:** una serie constante en los 49 meses (después del empalme) no tiene forma que comparar y se interpreta como dato congelado en la fuente. El artista se excluye, igual que los casos revisados a mano (`ARTISTAS_ANOMALIA_FUENTE`).
- **Quiebres comunes:** mes en que casi todos los artistas saltan en el mismo sentido. Se empalma la serie.
- **Faltantes:** interpolación lineal (`approx`, crecimiento a tasa constante). Antes del primer dato, el primer valor observado (`rule = 2`). Cada celda imputada queda marcada.

In [ ]:
# ============================================================
# LIMPIEZA DE PICOS AISLADOS E IMPUTACION DE FALTANTES
# ============================================================

panel_long[, log_valor := log1p(valor_crudo)]

panel_long[, pico_aislado := FALSE]

panel_long[
  variable %in% VARIABLES_STOCK_AUDIENCIA,
  pico_aislado := {
    salto_previo <- log_valor - shift(log_valor)
    salto_siguiente <- shift(log_valor, -1) - log_valor
    !is.na(salto_previo) & !is.na(salto_siguiente) &
      abs(salto_previo) > UMBRAL_SALTO_FUENTE &
      abs(salto_siguiente) > UMBRAL_SALTO_FUENTE &
      sign(salto_previo) != sign(salto_siguiente)
  },
  by = .(artist_id, variable)
]

panel_long[(pico_aislado), log_valor := NA_real_]

panel_long[, imputado := is.na(log_valor)]

panel_long[
  ,
  log_valor := approx(
    x = mes_indice[!imputado],
    y = log_valor[!imputado],
    xout = mes_indice,
    rule = 2
  )$y,
  by = .(artist_id, variable)
]

In [ ]:
# ------------------------------------------------------------
# CONTROL: sin faltantes despues de imputar, picos y celdas
# imputadas por variable, y detalle de los picos corregidos
# ------------------------------------------------------------

stopifnot(!anyNA(panel_long$log_valor))

panel_long[
  ,
  .(picos_aislados = sum(pico_aislado),
    celdas_imputadas = sum(imputado),
    pct_imputado = round(100 * mean(imputado), 2)),
  by = variable
]

panel_long[
  (pico_aislado),
  .(artist_name, variable, month, valor_crudo, valor_corregido = round(expm1(log_valor)))
]

In [ ]:
# ============================================================
# EXCLUSION DE ARTISTAS CON CAMBIOS DE NIVEL DE LA FUENTE
# (caida mensual mayor al umbral que no se revierte; los picos
# aislados ya fueron imputados y no cuentan como caida)
# ============================================================

series_con_cambio_nivel <- panel_long[
  variable %in% VARIABLES_STOCK_AUDIENCIA,
  .(caida_maxima = min(diff(log_valor)), mes_caida = month[which.min(diff(log_valor)) + 1]),
  by = .(artist_id, artist_name, variable)
][caida_maxima < -UMBRAL_SALTO_FUENTE]

stopifnot(!any(ARTISTAS_FIGURA_PRESENTACION %in% series_con_cambio_nivel$artist_id))

panel_long <- panel_long[!series_con_cambio_nivel, on = "artist_id"]
artistas_panel <- artistas_panel[!series_con_cambio_nivel, on = "artist_id"]

cat("Artistas excluidos:", uniqueN(series_con_cambio_nivel$artist_id),
    "| artistas en el panel:", nrow(artistas_panel), "\n")

series_con_cambio_nivel[order(caida_maxima), .(artist_name, variable, mes_caida, caida_maxima = round(caida_maxima, 2))]

In [ ]:
# ============================================================
# EMPALME EN QUIEBRES COMUNES DE LA FUENTE
# ============================================================

meses_quiebre_comun <- panel_long[
  ,
  .(month, cambio = log_valor - shift(log_valor)),
  by = .(artist_id, variable)
][
  !is.na(cambio),
  .(pct_suben = round(100 * mean(cambio > CAMBIO_MINIMO_QUIEBRE_COMUN), 1),
    pct_bajan = round(100 * mean(cambio < -CAMBIO_MINIMO_QUIEBRE_COMUN), 1)),
  by = .(variable, month)
][
  pmax(pct_suben, pct_bajan) >= 100 * PROPORCION_QUIEBRE_COMUN
]

panel_long[, empalme := FALSE]
panel_long[meses_quiebre_comun, empalme := TRUE, on = .(variable, month)]

# Cambio del mes de quiebre = mediana de los cambios del artista en
# los demas meses; se reconstruye la serie anclada en el ultimo mes
panel_long[
  variable %in% meses_quiebre_comun$variable,
  log_valor := {
    cambio <- c(0, diff(log_valor))
    cambio[empalme] <- median(cambio[-1][!empalme[-1]])
    serie_empalmada <- cumsum(cambio)
    serie_empalmada - serie_empalmada[.N] + log_valor[.N]
  },
  by = .(artist_id, variable)
]

meses_quiebre_comun

In [ ]:
# ------------------------------------------------------------
# CONTROL: mediana mensual del panel antes y despues del empalme
# en la variable afectada, alrededor de los meses de quiebre
# ------------------------------------------------------------

panel_long[
  variable %in% meses_quiebre_comun$variable & month <= max(meses_quiebre_comun$month) + 62,
  .(mediana_cruda = median(valor_crudo, na.rm = TRUE),
    mediana_empalmada = round(median(expm1(log_valor)), 1),
    empalme = any(empalme)),
  by = .(variable, month)
]

In [ ]:
# ============================================================
# EXCLUSION DE SERIES SIN VARIACION Y ANOMALIAS DE LA FUENTE
# Serie constante: log_valor sin variacion en los 49 meses,
# despues de imputar y empalmar (un unico salto en un mes de
# quiebre comun tambien la deja constante). Se excluye el
# artista completo: el clustering es multivariado.
# ============================================================

series_sin_variacion <- panel_long[
  ,
  .(desvio_log = sd(log_valor), valor_crudo_min = min(valor_crudo, na.rm = TRUE),
    valor_crudo_max = max(valor_crudo, na.rm = TRUE)),
  by = .(artist_id, artist_name, variable)
][desvio_log == 0]

artistas_excluidos_anomalia <- unique(rbind(
  series_sin_variacion[, .(artist_id, artist_name, motivo = paste("serie constante:", variable))],
  artistas_panel[artist_id %in% ARTISTAS_ANOMALIA_FUENTE, .(artist_id, artist_name, motivo = "anomalía revisada a mano")]
))

stopifnot(
  "Hay ids de ARTISTAS_ANOMALIA_FUENTE fuera del panel" =
    all(ARTISTAS_ANOMALIA_FUENTE %in% artistas_panel$artist_id),
  "Un artista de las figuras quedaría excluido" =
    !any(c(ARTISTAS_FIGURA_PRESENTACION, ARTISTAS_ILUSTRATIVOS) %in% artistas_excluidos_anomalia$artist_id)
)

panel_long <- panel_long[!artistas_excluidos_anomalia, on = "artist_id"]
artistas_panel <- artistas_panel[!artistas_excluidos_anomalia, on = "artist_id"]

cat("Artistas excluidos:", uniqueN(artistas_excluidos_anomalia$artist_id),
    "| artistas en el panel:", nrow(artistas_panel), "\n\n")

merge(
  artistas_excluidos_anomalia,
  series_sin_variacion[, .(artist_id, valor_crudo_min, valor_crudo_max)],
  by = "artist_id",
  all.x = TRUE
)

In [ ]:
# ============================================================
# FIGURA 2: un pico aislado corregido y el hueco interno mas
# largo del panel, interpolacion lineal vs spline
# ============================================================

pico_figura <- panel_long[(pico_aislado)][
  which.max(abs(log1p(valor_crudo) - log_valor))
]

huecos_internos <- panel_long[
  ,
  {
    tramos <- rle(imputado & !pico_aislado)
    fin_tramo <- cumsum(tramos$lengths)
    inicio_tramo <- fin_tramo - tramos$lengths + 1
    .(largo = tramos$lengths[tramos$values], inicio = inicio_tramo[tramos$values])
  },
  by = .(artist_id, variable)
][inicio > 1]

hueco_figura <- huecos_internos[order(-largo)][1]

grafico_pico_y_hueco <- function() {

  par(mfrow = c(1, 2), bg = COLOR_FONDO, mar = c(3, 4.5, 5.5, 1), las = 1, cex.axis = 0.85)

  # a) pico aislado
  serie_pico <- panel_long[.(pico_figura$artist_id, pico_figura$variable)]
  abrir_lienzo(range(serie_pico$valor_crudo, na.rm = TRUE),
               paste0("Pico aislado
", pico_figura$artist_name, ", ",
                      en_minuscula_inicial(VARIABLES_PANEL[[pico_figura$variable]])),
               escala_log = TRUE)
  lines(serie_pico$month, serie_pico$valor_crudo, col = COLOR_SERIE_BASE, lwd = 1.5)
  lines(serie_pico$month, expm1(serie_pico$log_valor), col = COLOR_RESULTADO, lwd = 2.5)
  points(pico_figura$month, pico_figura$valor_crudo, pch = 21, bg = COLOR_ALERTA, col = COLOR_FONDO, cex = 1.8)
  legend("bottomleft", c("serie cruda", "serie corregida", "pico aislado"), bty = "n", cex = 0.85,
         col = c(COLOR_SERIE_BASE, COLOR_RESULTADO, COLOR_ALERTA),
         lwd = c(1.5, 2.5, 0), lty = c(1, 1, 0), pch = c(NA, NA, 19), pt.cex = c(1, 1, 1.5))

  # b) hueco interno: interpolacion lineal (la usada) vs spline
  serie_hueco <- panel_long[.(hueco_figura$artist_id, hueco_figura$variable)]
  spline_hueco <- spline(serie_hueco[!(imputado), mes_indice], serie_hueco[!(imputado), log_valor],
                         xout = serie_hueco$mes_indice)$y
  abrir_lienzo(range(expm1(c(serie_hueco$log_valor, spline_hueco))),
               paste0("Hueco de ", hueco_figura$largo, " meses
",
                      serie_hueco$artist_name[1], ", ",
                      en_minuscula_inicial(VARIABLES_PANEL[[hueco_figura$variable]])))
  meses_hueco <- serie_hueco$month[hueco_figura$inicio + c(-1, hueco_figura$largo)]
  rect(meses_hueco[1], par("usr")[3], meses_hueco[2], par("usr")[4], col = "#F2F2F2", border = NA)
  lines(serie_hueco$month, expm1(spline_hueco), col = COLOR_ALERTA, lwd = 2, lty = 2)
  lines(serie_hueco$month, expm1(serie_hueco$log_valor), col = COLOR_RESULTADO, lwd = 2.5)
  points(serie_hueco[!(imputado), month], serie_hueco[!(imputado), valor_crudo], pch = 19, col = COLOR_SERIE_BASE, cex = 0.9)
  legend("bottom", c("dato observado", "interpolación lineal", "spline cúbica"), bty = "n", cex = 0.85,
         col = c(COLOR_SERIE_BASE, COLOR_RESULTADO, COLOR_ALERTA),
         lwd = c(NA, 2.5, 2), lty = c(NA, 1, 2), pch = c(19, NA, NA))
}

guardar_figura("prepro_03_pico_y_hueco.png", grafico_pico_y_hueco, ancho = 2600, alto = 1000)

## 5. Transformación

`log1p` y centrado por serie (artista y variable). El log hace comparable el crecimiento entre niveles; el centrado quita el nivel y deja la forma, para no agrupar por popularidad. El nivel queda en `log_valor`. Alternativas en `TRANSFORMACION`.

In [ ]:
# ============================================================
# TRANSFORMACION DE LAS SERIES
# ============================================================

transformar_serie <- function(log_valor, metodo = TRANSFORMACION) {
  switch(
    metodo,
    log1p_centrado      = log_valor - mean(log_valor),
    log1p_estandarizado = if (sd(log_valor) > 0) (log_valor - mean(log_valor)) / sd(log_valor)
                          else log_valor - mean(log_valor),
    log1p_desde_inicio  = log_valor - log_valor[1],
    sin_transformar     = expm1(log_valor),
    stop("TRANSFORMACION no reconocida: ", metodo)
  )
}

panel_long[
  ,
  valor_transformado := transformar_serie(log_valor),
  by = .(artist_id, variable)
]

In [ ]:
# ------------------------------------------------------------
# CONTROL: media de cada serie transformada (cero con el
# centrado) y amplitud de las trayectorias por variable
# (maximo menos minimo, en unidades log)
# ------------------------------------------------------------

amplitud_series <- panel_long[
  ,
  .(media = mean(valor_transformado), amplitud = diff(range(valor_transformado))),
  by = .(artist_id, variable)
]

cat("Rango de las medias por serie:", signif(range(amplitud_series$media), 3), "\n")

amplitud_series[, as.list(round(quantile(amplitud, c(0.1, 0.5, 0.9)), 2)), by = variable]

In [ ]:
# ============================================================
# FIGURA 3: los mismos artistas despues de la transformacion
# ============================================================

guardar_figura(
  "prepro_04_series_transformadas.png",
  function() grafico_cinco_variables(
    "valor_transformado",
    "Series transformadas: log1p centrado por artista y variable (punteado: meses imputados)",
    escala_log = FALSE
  ),
  ancho = 3000, alto = 950
)

In [ ]:
# ============================================================
# FIGURA 3b: distribucion de una variable entre artistas en un
# mes, en las tres etapas: cruda, log1p y log1p centrado por
# artista. Ed Sheeran marcado como referencia
# ============================================================

VARIABLE_DISTRIBUCION <- "spotify_followers"
ARTISTA_DISTRIBUCION <- ARTISTAS_FIGURA_PRESENTACION[1]
COLOR_ARTISTA_DISTRIBUCION <- PALETA_ARTISTAS[1]

# Mes con mayor cobertura de datos observados en la variable (el
# mas reciente si hay empate). Solo artistas del panel final.
cobertura_por_mes <- panel_long[
  variable == VARIABLE_DISTRIBUCION,
  .(observados = sum(!is.na(valor_crudo))),
  by = month
]
MES_DISTRIBUCION <- cobertura_por_mes[observados == max(observados), max(month)]

corte_distribucion <- panel_long[
  variable == VARIABLE_DISTRIBUCION & month == MES_DISTRIBUCION & !is.na(valor_crudo),
  .(artist_id, cruda = valor_crudo, log1p = log_valor, centrada = valor_transformado)
]

cat(
  "Mes:", format(MES_DISTRIBUCION), "| artistas con dato:", nrow(corte_distribucion),
  "de", uniqueN(panel_long$artist_id), "\n"
)

ETAPAS_DISTRIBUCION <- c(
  cruda = "Cruda",
  log1p = "log1p",
  centrada = "log1p centrado por artista"
)

TAMANIO_TITULO_DISTRIBUCION <- 2.4
TAMANIO_TITULO_PANEL_DISTRIBUCION <- 1.5
TAMANIO_TEXTO_DISTRIBUCION <- 1.5

grafico_densidad_etapas <- function() {

  # con 3 paneles en fila R reduce los textos a 2/3: los tamanios
  # de arriba lo compensan
  par(mfrow = c(1, length(ETAPAS_DISTRIBUCION)), bg = COLOR_FONDO, mar = c(4.5, 2, 4, 2),
      oma = c(0, 0, 6, 0), las = 1, cex.axis = TAMANIO_TEXTO_DISTRIBUCION)

  for (etapa in names(ETAPAS_DISTRIBUCION)) {

    valores <- corte_distribucion[[etapa]]
    densidad <- density(valores, n = 1024, from = if (etapa == "cruda") 0 else min(valores),
                        to = max(valores))
    valor_artista <- corte_distribucion[artist_id == ARTISTA_DISTRIBUCION, get(etapa)]

    plot(densidad$x, densidad$y, type = "n", bty = "n", xlab = "", ylab = "", yaxt = "n", xaxt = "n",
         ylim = c(0, max(densidad$y) * 1.15))
    title(ETAPAS_DISTRIBUCION[[etapa]], font.main = 1, cex.main = TAMANIO_TITULO_PANEL_DISTRIBUCION, line = 1.5)

    marcas_x <- axTicks(1)
    axis(1, at = marcas_x, col = NA, col.ticks = "#BDBDBD",
         labels = if (etapa == "cruda") etiqueta_eje_magnitud(marcas_x) else format(marcas_x, decimal.mark = ","))

    polygon(c(densidad$x, rev(densidad$x)), c(densidad$y, rep(0, length(densidad$y))),
            col = COLOR_SERIE_BASE, border = NA)
    lines(densidad$x, densidad$y, col = COLOR_RESULTADO, lwd = 2.5)

    abline(v = valor_artista, col = COLOR_ARTISTA_DISTRIBUCION, lwd = 4)
    text(valor_artista, max(densidad$y) * 1.1, artistas_panel[artist_id == ARTISTA_DISTRIBUCION, artist_name],
         pos = if (valor_artista > mean(range(densidad$x))) 2 else 4,
         col = COLOR_ARTISTA_DISTRIBUCION, cex = TAMANIO_TEXTO_DISTRIBUCION, font = 2)
  }

  mtext(paste0(VARIABLES_PANEL[[VARIABLE_DISTRIBUCION]], ", ", format(MES_DISTRIBUCION, "%B %Y"),
               ": distribución entre los ", nrow(corte_distribucion), " artistas en cada etapa"),
        side = 3, outer = TRUE, line = 3, font = 2, cex = TAMANIO_TITULO_DISTRIBUCION, family = "LWR Titulo")
  mtext("Curva de densidad: dónde se concentran los artistas. Un valor por artista.",
        side = 3, outer = TRUE, line = 0.8, cex = 1.2, col = COLOR_REFERENCIA)
}

guardar_figura("prepro_04b_distribucion_etapas.png", grafico_densidad_etapas, ancho = 3200, alto = 1100)


Cruda, la distribución es muy asimétrica; con `log1p` se vuelve aproximadamente simétrica; centrada, mide cuánto se aparta cada artista de su propia trayectoria.

## 6. Estacionaridad

ADF (H0: raíz unitaria) y KPSS (H0: estacionaria) sobre cada serie transformada y su primera diferencia.

In [ ]:
# ============================================================
# PRUEBAS ADF Y KPSS POR SERIE: transformada y primera diferencia
# ============================================================

decidir_estacionaridad <- function(serie) {
  p_valores <- tryCatch(
    suppressWarnings(c(adf = adf.test(serie)$p.value, kpss = kpss.test(serie)$p.value)),
    error = function(e) c(adf = NA, kpss = NA)
  )
  fcase(
    p_valores[["adf"]] < 0.05 & p_valores[["kpss"]] >= 0.05, "estacionaria",
    p_valores[["adf"]] >= 0.05 & p_valores[["kpss"]] < 0.05, "no estacionaria",
    default = "indefinida"
  )
}

estacionaridad_series <- panel_long[
  ,
  .(transformada = decidir_estacionaridad(valor_transformado),
    primera_diferencia = decidir_estacionaridad(diff(valor_transformado))),
  by = .(artist_id, variable)
]

resumen_estacionaridad <- melt(
  estacionaridad_series,
  id.vars = c("artist_id", "variable"),
  variable.name = "version",
  value.name = "decision"
)[
  , .N, by = .(version, variable, decision)
][
  , pct := round(100 * N / sum(N), 1), by = .(version, variable)
]

dcast(resumen_estacionaridad, version + variable ~ decision, value.var = "pct", fill = 0)

In [ ]:
# ============================================================
# FIGURA 5b: estacionaridad en lenguaje llano. Izquierda: que es
# la primera diferencia, con un artista. Derecha: decision de
# ADF y KPSS por variable, en nivel y en primera diferencia
# ============================================================

ARTISTA_EJEMPLO_ESTACIONARIDAD <- 3648L   # Ed Sheeran
VARIABLE_EJEMPLO_ESTACIONARIDAD <- "spotify_followers"

# Escala sin rojo: no estacionaria en negro, indefinida en gris y
# estacionaria en blanco con borde (la decisión "más quieta")
COLORES_DECISION <- c("no estacionaria" = "#111111", "indefinida" = "#BDBDBD",
                      "estacionaria" = "#FFFFFF")
COLOR_TEXTO_DECISION <- c("no estacionaria" = "#FFFFFF", "indefinida" = "#111111",
                          "estacionaria" = "#111111")
ETIQUETAS_DECISION <- c("no estacionaria" = "cambia con el tiempo",
                        "indefinida" = "las pruebas no coinciden",
                        "estacionaria" = "estable")
ETIQUETAS_VERSION <- c(transformada = "Lo acumulado (nivel)",
                       primera_diferencia = "Lo que se suma cada mes (primera diferencia)")

serie_ejemplo <- panel_long[artist_id == ARTISTA_EJEMPLO_ESTACIONARIDAD & variable == VARIABLE_EJEMPLO_ESTACIONARIDAD]
nombre_ejemplo <- artistas_panel[artist_id == ARTISTA_EJEMPLO_ESTACIONARIDAD, artist_name]

# Matriz decision x barra: una barra por version y variable, en %
orden_barras <- CJ(version = names(ETIQUETAS_VERSION), variable = names(VARIABLES_PANEL), sorted = FALSE)
matriz_decision <- sapply(seq_len(nrow(orden_barras)), function(i) {
  fila <- resumen_estacionaridad[orden_barras[i], on = .(version, variable)]
  setNames(fila$pct, fila$decision)[names(COLORES_DECISION)]
})
matriz_decision[is.na(matriz_decision)] <- 0

# Sanity check: cada barra suma 100%
stopifnot(all(abs(colSums(matriz_decision) - 100) < 0.5))

grafico_estacionaridad <- function() {

  layout(matrix(c(1, 2, 3, 3), nrow = 2), widths = c(1, 1.3))
  par(bg = COLOR_FONDO, las = 1, cex.axis = 1.1)

  # a) nivel: los seguidores acumulados suben, la media cambia
  par(mar = c(3.5, 5, 6.5, 4))
  abrir_lienzo(range(serie_ejemplo$valor_transformado), "")
  lines(serie_ejemplo$month, serie_ejemplo$valor_transformado, col = COLOR_RESULTADO, lwd = 3)
  mtext(paste0(nombre_ejemplo, ", ", en_minuscula_inicial(VARIABLES_PANEL[[VARIABLE_EJEMPLO_ESTACIONARIDAD]])),
        side = 3, line = 4, adj = 0, font = 2, cex = 1.2)
  mtext("Lo acumulado: sube todo el período,\nsu promedio cambia", side = 3, line = 0.8, adj = 0, cex = 1.05,
        col = COLOR_REFERENCIA)

  # b) primera diferencia: lo que suma cada mes, en % de crecimiento
  crecimiento_mensual_pct <- 100 * diff(serie_ejemplo$valor_transformado)
  par(mar = c(3.5, 5, 5, 4))
  abrir_lienzo(range(0, crecimiento_mensual_pct), "")
  segments(serie_ejemplo$month[-1], 0, serie_ejemplo$month[-1], crecimiento_mensual_pct,
           col = COLOR_RESULTADO, lwd = 4, lend = 1)
  mtext("Lo que se suma cada mes (% de crecimiento):\nmás quieto, pero se frena", side = 3, line = 0.8, adj = 0,
        cex = 1.05, col = COLOR_REFERENCIA)

  # c) decision por variable, barras apiladas al 100%
  par(mar = c(7, 16, 3, 2))
  espacios <- c(0.35, rep(0.3, 4), 1.9, rep(0.3, 4))   # orden de dibujo: de abajo hacia arriba
  posiciones <- barplot(matriz_decision[, rev(seq_len(ncol(matriz_decision)))], horiz = TRUE,
                        col = COLORES_DECISION, border = "#111111", lwd = 1, space = espacios,
                        names.arg = rev(VARIABLES_PANEL[orden_barras$variable]), xlim = c(0, 100),
                        axes = FALSE, cex.names = 1.15)
  posiciones <- rev(posiciones)
  axis(1, at = seq(0, 100, 25), labels = paste0(seq(0, 100, 25), "%"), col = NA, col.ticks = COLOR_GRILLA)
  # porcentaje dentro de cada tramo con espacio suficiente, en el color que contrasta
  inicio_tramo <- apply(matriz_decision, 2, function(x) cumsum(x) - x)
  for (j in seq_len(ncol(matriz_decision))) {
    visibles <- matriz_decision[, j] >= 7
    text(inicio_tramo[visibles, j] + matriz_decision[visibles, j] / 2, posiciones[j],
         paste0(round(matriz_decision[visibles, j]), "%"), cex = 1.35, font = 2,
         col = COLOR_TEXTO_DECISION[names(COLORES_DECISION)[visibles]])
  }
  # titulo de cada grupo sobre su primera barra
  for (version_grupo in names(ETIQUETAS_VERSION)) {
    primera_barra <- which(orden_barras$version == version_grupo)[1]
    text(0, posiciones[primera_barra] + 1.05, ETIQUETAS_VERSION[[version_grupo]], adj = c(0, 0),
         xpd = TRUE, font = 2, cex = 1.25)
  }
  par(xpd = NA)
  legend("bottom", inset = c(0, -0.15), horiz = TRUE, bty = "n", fill = COLORES_DECISION, border = "#111111",
         legend = ETIQUETAS_DECISION, cex = 1.05, x.intersp = 0.6, text.width = NA)
}

guardar_figura("prepro_04c_estacionaridad.png", grafico_estacionaridad, ancho = 3400, alto = 1800)

Casi todas las audiencias son no estacionarias y la primera diferencia no lo resuelve del todo: la tasa de crecimiento cambia en el tiempo. Se trabaja en nivel (log centrado) para comparar formas; el crecimiento se calcula aparte (bloque 9).

## 7. Suavizado por media móvil

Media móvil centrada de **3 meses** (ruido mes a mes) y **2x12 meses** (tendencia anual, el filtro de `decompose()`). Sin filtro circular; en los bordes, ventana simétrica más corta.

In [ ]:
# ============================================================
# MEDIA MOVIL CENTRADA SOBRE TODO EL PANEL
# ============================================================

stopifnot(
  VENTANAS_MEDIA_MOVIL[["suavizado"]] %% 2 == 1,
  VENTANAS_MEDIA_MOVIL[["tendencia"]] %% 2 == 0
)

# Media movil simple centrada, ventana impar
media_movil_centrada <- function(x, ventana) {
  largo_serie <- length(x)
  posiciones <- seq_len(largo_serie)
  semiancho <- pmin((ventana - 1) %/% 2, posiciones - 1, largo_serie - posiciones)
  sumas_acumuladas <- c(0, cumsum(x))
  (sumas_acumuladas[posiciones + semiancho + 1] - sumas_acumuladas[posiciones - semiancho]) /
    (2 * semiancho + 1)
}

# Media movil 2xn centrada (n par): n + 1 coeficientes, extremos a
# mitad de peso. En los bordes, media simple simetrica mas corta
media_movil_2xn <- function(x, n) {
  pesos <- c(0.5, rep(1, n - 1), 0.5) / n
  tendencia <- as.numeric(stats::filter(x, pesos, sides = 2))
  bordes <- is.na(tendencia)
  tendencia[bordes] <- media_movil_centrada(x, n + 1)[bordes]
  tendencia
}

panel_long[
  ,
  `:=`(
    valor_suavizado = media_movil_centrada(valor_transformado, VENTANAS_MEDIA_MOVIL[["suavizado"]]),
    tendencia = media_movil_2xn(valor_transformado, VENTANAS_MEDIA_MOVIL[["tendencia"]])
  ),
  by = .(artist_id, variable)
]

In [ ]:
# ------------------------------------------------------------
# CONTROL: lejos de los bordes, el suavizado coincide con filter()
# de la clase (no circular) y la tendencia con la de decompose()
# ------------------------------------------------------------

serie_control <- panel_long[.(ARTISTAS_FIGURA_PRESENTACION[1], VARIABLE_ILUSTRATIVA)]

suavizado_clase <- as.numeric(stats::filter(
  serie_control$valor_transformado,
  rep(1 / VENTANAS_MEDIA_MOVIL[["suavizado"]], VENTANAS_MEDIA_MOVIL[["suavizado"]]),
  sides = 2
))

tendencia_decompose <- as.numeric(decompose(
  ts(serie_control$valor_transformado, frequency = VENTANAS_MEDIA_MOVIL[["tendencia"]])
)$trend)

cat(
  "Suavizado igual a filter() en el interior:",
  isTRUE(all.equal(suavizado_clase[!is.na(suavizado_clase)],
                   serie_control$valor_suavizado[!is.na(suavizado_clase)])), "\n",
  "Tendencia igual a decompose() en el interior:",
  isTRUE(all.equal(tendencia_decompose[!is.na(tendencia_decompose)],
                   serie_control$tendencia[!is.na(tendencia_decompose)])), "\n"
)

Ejemplos: un artista por tercil de `cm_artist_rank`, sin meses imputados.

In [ ]:
# ============================================================
# ARTISTAS ILUSTRATIVOS: tercil de rank y control de cobertura
# ============================================================

# Terciles por posicion y no por valor de rank, para no depender
# de cortes que caigan en valores repetidos
artistas_panel[
  ,
  tercil_rank := cut(
    frank(cm_artist_rank, ties.method = "first"),
    breaks = 3,
    labels = c("rank_alto", "rank_medio", "rank_bajo")
  )
]

ids_ilustrativos <- unname(ARTISTAS_ILUSTRATIVOS)

# Mismo orden que ARTISTAS_ILUSTRATIVOS (alto, medio, bajo)
artistas_ilustrativos <- artistas_panel[match(ids_ilustrativos, artist_id)]

artistas_ilustrativos[
  panel_long[artist_id %in% ids_ilustrativos, .(meses_imputados = sum(imputado)), by = artist_id],
  meses_imputados := i.meses_imputados,
  on = "artist_id"
]

stopifnot(
  "Algun artista ilustrativo no esta en el panel" = !anyNA(artistas_ilustrativos$artist_name),
  "Algun artista ilustrativo no cae en su tercil" =
    all(as.character(artistas_ilustrativos$tercil_rank) == names(ARTISTAS_ILUSTRATIVOS)),
  "Algun artista ilustrativo tiene meses imputados" = all(artistas_ilustrativos$meses_imputados == 0)
)

artistas_ilustrativos[, .(tercil_rank, artist_id, artist_name, cm_artist_rank)]

In [ ]:
# ============================================================
# PANEL 3 x 5 (no se genera): serie transformada, suavizado y
# tendencia para los tres artistas ilustrativos y las cinco
# variables. Se conserva para uso posterior
# ============================================================

grafico_suavizado <- function() {

  par(mfrow = c(length(ids_ilustrativos), length(VARIABLES_PANEL)), bg = COLOR_FONDO,
      mar = c(2.5, 3.8, 2.2, 0.8), oma = c(0, 3, 5, 0), las = 1, cex.axis = 0.8)

  for (i in seq_along(ids_ilustrativos)) {
    for (variable_figura in names(VARIABLES_PANEL)) {
      serie_artista <- panel_long[.(ids_ilustrativos[i], variable_figura)]
      abrir_lienzo(range(serie_artista$valor_transformado),
                   if (i == 1) VARIABLES_PANEL[[variable_figura]] else "")
      lines(serie_artista$month, serie_artista$valor_transformado, col = COLOR_SERIE_BASE, lwd = 1.5)
      lines(serie_artista$month, serie_artista$valor_suavizado, col = COLOR_MEDIA_MOVIL, lwd = 2)
      lines(serie_artista$month, serie_artista$tendencia, col = COLOR_TENDENCIA, lwd = 2)
    }
    mtext(paste0(artistas_ilustrativos$artist_name[i], " (rank ", artistas_ilustrativos$cm_artist_rank[i], ")"),
          side = 2, outer = TRUE, line = 1, las = 0, cex = 0.95,
          at = 1 - (i - 0.5) / length(ids_ilustrativos))
  }

  mtext("Suavizado por media móvil centrada",
        side = 3, outer = TRUE, line = 3, cex = 1.9, family = "LWR Titulo")

  par(fig = c(0, 1, 0, 1), oma = c(0, 0, 0, 0), mar = c(0, 0, 0, 0), new = TRUE)
  plot(0, 0, type = "n", bty = "n", xaxt = "n", yaxt = "n")
  legend("top", horiz = TRUE, bty = "n", inset = c(0, 0.045), cex = 0.95, lwd = c(1.5, 2, 2),
         col = c(COLOR_SERIE_BASE, COLOR_MEDIA_MOVIL, COLOR_TENDENCIA),
         legend = c("transformada",
                    paste0("media móvil ", VENTANAS_MEDIA_MOVIL[["suavizado"]], " meses"),
                    paste0("tendencia (media móvil 2×", VENTANAS_MEDIA_MOVIL[["tendencia"]], ")")))
}

# guardar_figura("prepro_05_suavizado_media_movil_panel.png", grafico_suavizado, ancho = 3000, alto = 1900)

In [ ]:
# ============================================================
# FIGURA 5: serie transformada, suavizado y tendencia de un
# artista ilustrativo, una variable lenta y una ruidosa
# ============================================================

ARTISTA_FIGURA_SUAVIZADO <- ARTISTAS_ILUSTRATIVOS[["rank_alto"]]
VARIABLES_FIGURA_SUAVIZADO <- c("spotify_followers", "youtube_playlist_count")

# Tamanios de texto: titulo general > titulo de cada panel > ejes y leyenda
TAMANIO_TITULO_FIGURA_5 <- 2.4
TAMANIO_TITULO_PANEL_FIGURA_5 <- 1.5
TAMANIO_TEXTO_FIGURA_5 <- 1.3

grafico_suavizado_artista <- function(
  artista = ARTISTA_FIGURA_SUAVIZADO,
  variables = VARIABLES_FIGURA_SUAVIZADO,
  margen_leyenda = 0.075,
  colores = c(transformada = COLOR_SERIE_BASE, media_movil = COLOR_MEDIA_MOVIL, tendencia = COLOR_TENDENCIA)
) {

  par(mfrow = c(length(variables), 1), bg = COLOR_FONDO,
      mar = c(3.5, 5.5, 4, 1), oma = c(0, 0, 7, 0), las = 1, cex.axis = TAMANIO_TEXTO_FIGURA_5)

  for (variable_figura in variables) {
    serie_artista <- panel_long[.(artista, variable_figura)]
    abrir_lienzo(range(serie_artista$valor_transformado), "")
    title(paste0(serie_artista$artist_name[1], ": ", en_minuscula_inicial(VARIABLES_PANEL[[variable_figura]])),
          font.main = 1, cex.main = TAMANIO_TITULO_PANEL_FIGURA_5, line = 1.5)
    # una marca y un punto por mes para que se vea la granularidad mensual
    axis(1, at = MESES_PANEL, labels = FALSE, col = NA, col.ticks = "#BDBDBD", tcl = -0.3)
    lines(serie_artista$month, serie_artista$valor_transformado, col = colores[["transformada"]], lwd = 2)
    points(serie_artista$month, serie_artista$valor_transformado, pch = 19, col = colores[["transformada"]], cex = 1.1)
    lines(serie_artista$month, serie_artista$valor_suavizado, col = colores[["media_movil"]], lwd = 4)
    lines(serie_artista$month, serie_artista$tendencia, col = colores[["tendencia"]], lwd = 4)
  }

  mtext("Suavizado por media móvil centrada",
        side = 3, outer = TRUE, line = 4.2, font = 2, cex = TAMANIO_TITULO_FIGURA_5, family = "LWR Titulo")

  par(fig = c(0, 1, 0, 1), oma = c(0, 0, 0, 0), mar = c(0, 0, 0, 0), new = TRUE)
  plot(0, 0, type = "n", bty = "n", xaxt = "n", yaxt = "n")
  legend("top", horiz = TRUE, bty = "n", inset = c(0, margen_leyenda), cex = TAMANIO_TEXTO_FIGURA_5,
         lwd = c(2, 4, 4), pch = c(19, NA, NA), pt.cex = 1.1, seg.len = 2.5,
         col = colores[c("transformada", "media_movil", "tendencia")],
         legend = c("transformada",
                    paste0("media móvil ", VENTANAS_MEDIA_MOVIL[["suavizado"]], " meses"),
                    paste0("tendencia (media móvil 2×", VENTANAS_MEDIA_MOVIL[["tendencia"]], ")")))
}

guardar_figura("prepro_05_suavizado_media_movil.png", grafico_suavizado_artista, ancho = 3200, alto = 1800)


In [ ]:
# ============================================================
# FIGURA 5, VERSION EN TONOS DEL ARTISTA: la figura 5 para el
# primer artista de la figura de presentacion, con tres tonos de
# su color (claro, propio y oscuro) en lugar de la codificacion
# general. Se guarda con el nombre del artista como sufijo.
# ============================================================

ARTISTA_VERSION_TONOS <- ARTISTAS_FIGURA_PRESENTACION[1]

COLOR_ARTISTA_VERSION_TONOS <- PALETA_ARTISTAS[match(ARTISTA_VERSION_TONOS, ARTISTAS_FIGURA_PRESENTACION)]

SUFIJO_VERSION_TONOS <- gsub(
  "[^a-z0-9]+", "_",
  tolower(artistas_panel[artist_id == ARTISTA_VERSION_TONOS, artist_name])
)

# Tres tonos de un mismo color: mezcla con blanco, el color
# original y mezcla con negro
tonos_de_color <- function(color) {
  escala <- colorRampPalette(c("#FFFFFF", color, "#000000"))(9)
  c(claro = escala[3], propio = escala[5], oscuro = escala[7])
}

TONOS_VERSION_ARTISTA <- tonos_de_color(COLOR_ARTISTA_VERSION_TONOS)

# Serie transformada en el color propio del artista, el mismo de
# la figura 4; media movil de 3 meses en gris claro, que contrasta
# con ese color y con el tono oscuro de la tendencia
COLOR_MEDIA_MOVIL_VERSION_TONOS <- "#BDBDBD"

guardar_figura(
  paste0("prepro_05_suavizado_media_movil_", SUFIJO_VERSION_TONOS, ".png"),
  function() grafico_suavizado_artista(
    artista = ARTISTA_VERSION_TONOS,
    colores = c(transformada = TONOS_VERSION_ARTISTA[["propio"]],
                media_movil = COLOR_MEDIA_MOVIL_VERSION_TONOS,
                tendencia = TONOS_VERSION_ARTISTA[["oscuro"]])
  ),
  ancho = 3200, alto = 1800
)


In [ ]:
# ============================================================
# FIGURA 5, SOLO PLAYLISTS EN YOUTUBE: el panel ruidoso de la
# version en tonos, que es donde se ve el costo de la ventana
# ancha (la caida de fines de 2024). Mismas leyendas y colores
# ============================================================

guardar_figura(
  paste0("prepro_05_suavizado_media_movil_", SUFIJO_VERSION_TONOS, "_youtube.png"),
  function() grafico_suavizado_artista(
    artista = ARTISTA_VERSION_TONOS,
    variables = "youtube_playlist_count",
    margen_leyenda = 0.13,
    colores = c(transformada = TONOS_VERSION_ARTISTA[["propio"]],
                media_movil = COLOR_MEDIA_MOVIL_VERSION_TONOS,
                tendencia = TONOS_VERSION_ARTISTA[["oscuro"]])
  ),
  ancho = 3200, alto = 1100
)


El insumo del clustering es la serie suavizada a 3 meses: la tendencia 2x12 borra cambios reales, como la caída de Abel Pintos en Spotify a fines de 2023.

## 8. Dominio de la frecuencia

FFT de cada serie sin su tendencia lineal (la FFT supone que la serie se repite). Armónico k: k ciclos en 49 meses; k = 24 es el límite de Nyquist.

In [ ]:
# ============================================================
# ESPECTRO DE AMPLITUD POR SERIE
# ============================================================

N_ARMONICOS <- N_MESES_PANEL %/% 2
CICLOS_POR_ANIO_ARMONICOS <- seq_len(N_ARMONICOS) * 12 / N_MESES_PANEL

# Amplitud de cada armonico k = 1..N/2 de la serie sin su recta de
# tendencia. 2 * Mod / N es la amplitud de la oscilacion (la FFT
# reparte cada oscilacion entre k y su espejo N - k)
espectro_amplitud <- function(x) {
  residuo <- residuals(lm(x ~ seq_along(x)))
  2 * Mod(fft(residuo))[2:(N_ARMONICOS + 1)] / length(x)
}

espectro_panel <- panel_long[
  ,
  .(armonico = seq_len(N_ARMONICOS),
    ciclos_por_anio = CICLOS_POR_ANIO_ARMONICOS,
    amplitud_transformada = espectro_amplitud(valor_transformado),
    amplitud_suavizada = espectro_amplitud(valor_suavizado)),
  by = .(artist_id, variable)
]

In [ ]:
# ============================================================
# FIGURA 6: espectro de amplitud de los artistas ilustrativos,
# una variable lenta y una ruidosa
# ============================================================

VARIABLES_FIGURA_ESPECTRO <- c("spotify_followers", "youtube_playlist_count")

grafico_espectro_amplitud <- function() {

  par(mfrow = c(length(VARIABLES_FIGURA_ESPECTRO), length(ids_ilustrativos)), bg = COLOR_FONDO,
      mar = c(4.5, 4, 4.5, 1), oma = c(0, 3, 5, 0), las = 1, cex.axis = 0.85)

  for (variable_figura in VARIABLES_FIGURA_ESPECTRO) {
    for (i in seq_along(ids_ilustrativos)) {

      espectro_artista <- espectro_panel[artist_id == ids_ilustrativos[i] & variable == variable_figura]

      plot(espectro_artista$ciclos_por_anio, espectro_artista$amplitud_transformada,
           type = "n", bty = "n", xlab = "Ciclos por año", ylab = "", yaxt = "n",
           ylim = c(0, max(espectro_artista$amplitud_transformada)), xlim = c(0, 6))
      marcas_y <- axTicks(2)
      axis(2, at = marcas_y, labels = format(marcas_y, decimal.mark = ","), col = NA)
      abline(h = marcas_y, col = COLOR_GRILLA)

      # eje superior: periodo de la oscilacion en meses
      periodos_meses <- c(48, 12, 6, 4, 3, 2)
      axis(3, at = 12 / periodos_meses, labels = periodos_meses, col = NA, col.ticks = "#BDBDBD", cex.axis = 0.75)
      mtext("período (meses)", side = 3, line = 2, cex = 0.6, col = "#6B6B6B")

      segments(espectro_artista$ciclos_por_anio, 0, espectro_artista$ciclos_por_anio,
               espectro_artista$amplitud_transformada, col = COLOR_SERIE_BASE, lwd = 7, lend = 1)
      segments(espectro_artista$ciclos_por_anio, 0, espectro_artista$ciclos_por_anio,
               espectro_artista$amplitud_suavizada, col = COLOR_MEDIA_MOVIL, lwd = 2.5, lend = 1)

      title(paste0(artistas_ilustrativos$artist_name[i], ": ", en_minuscula_inicial(VARIABLES_PANEL[[variable_figura]])),
            font.main = 1, cex.main = 1, line = 3.3)
    }
  }

  mtext("Espectro de amplitud (FFT de la serie sin tendencia lineal)", side = 3, outer = TRUE, line = 3.2, cex = 1.9, family = "LWR Titulo")
  mtext("amplitud (unidades log)", side = 2, outer = TRUE, line = 1, las = 0, cex = 0.85)

  par(fig = c(0, 1, 0, 1), oma = c(0, 0, 0, 0), mar = c(0, 0, 0, 0), new = TRUE)
  plot(0, 0, type = "n", bty = "n", xaxt = "n", yaxt = "n")
  legend("top", horiz = TRUE, bty = "n", inset = c(0, 0.025), cex = 0.9, lwd = c(7, 2.5),
         col = c(COLOR_SERIE_BASE, COLOR_MEDIA_MOVIL),
         legend = c("serie transformada", paste0("media móvil ", VENTANAS_MEDIA_MOVIL[["suavizado"]], " meses")))
}

guardar_figura("prepro_06_espectro_amplitud.png", grafico_espectro_amplitud, ancho = 2800, alto = 1800)

La amplitud se concentra en los primeros armónicos, salvo en las playlists de YouTube. La tabla resume la energía en oscilaciones de un año o más y de menos de 6 meses.

In [ ]:
# ------------------------------------------------------------
# ENERGIA POR BANDA DE FRECUENCIA, MEDIANA POR VARIABLE (%)
# (las series constantes no tienen energia y quedan fuera)
# ------------------------------------------------------------

espectro_panel[
  ,
  .(lenta_transformada = sum(amplitud_transformada[ciclos_por_anio <= 1]^2) / sum(amplitud_transformada^2),
    rapida_transformada = sum(amplitud_transformada[ciclos_por_anio > 2]^2) / sum(amplitud_transformada^2),
    rapida_suavizada = sum(amplitud_suavizada[ciclos_por_anio > 2]^2) / sum(amplitud_transformada^2)),
  by = .(artist_id, variable)
][
  ,
  lapply(.SD, function(x) round(100 * median(x, na.rm = TRUE), 1)),
  by = variable,
  .SDcols = c("lenta_transformada", "rapida_transformada", "rapida_suavizada")
]

**Respuesta en frecuencia** de las medias móviles (corte en 0,7) y **pasa-bajos FFT** sobre un artista ilustrativo, con y sin restar la tendencia.

In [ ]:
# ============================================================
# RESPUESTA EN FRECUENCIA DE LAS MEDIAS MOVILES Y PASA-BAJOS FFT
# ============================================================

# Coeficientes de cada filtro completados con ceros hasta un largo
# grande, para obtener una curva continua de |H(f)|
LARGO_RESPUESTA_FRECUENCIA <- 1200

respuesta_en_frecuencia <- function(coeficientes) {
  Mod(fft(c(coeficientes, rep(0, LARGO_RESPUESTA_FRECUENCIA - length(coeficientes)))))[
    1:(LARGO_RESPUESTA_FRECUENCIA %/% 2 + 1)
  ]
}

respuesta_filtros <- data.table(
  ciclos_por_anio = (0:(LARGO_RESPUESTA_FRECUENCIA %/% 2)) * 12 / LARGO_RESPUESTA_FRECUENCIA,
  suavizado = respuesta_en_frecuencia(rep(1 / VENTANAS_MEDIA_MOVIL[["suavizado"]], VENTANAS_MEDIA_MOVIL[["suavizado"]])),
  tendencia = respuesta_en_frecuencia(c(0.5, rep(1, VENTANAS_MEDIA_MOVIL[["tendencia"]] - 1), 0.5) / VENTANAS_MEDIA_MOVIL[["tendencia"]])
)

# Frecuencia donde cada filtro cae por debajo de 0,7
cortes_media_movil <- respuesta_filtros[
  , .(corte_suavizado = ciclos_por_anio[which(suavizado < 0.7)[1]],
      corte_tendencia = ciclos_por_anio[which(tendencia < 0.7)[1]])
]

cortes_media_movil

# Mascara pasa-bajos que conserva la frecuencia cero, los primeros
# armonicos y su espejo (indices N - k + 1)
filtro_pasa_bajos_fft <- function(x, banda = BANDA_FILTRO_PASA_BAJOS, quitar_tendencia = TRUE) {
  largo_serie <- length(x)
  recta <- if (quitar_tendencia) fitted(lm(x ~ seq_along(x))) else rep(0, largo_serie)
  mascara <- rep(0, largo_serie)
  mascara[1:(banda + 1)] <- 1
  mascara[(largo_serie - banda + 1):largo_serie] <- 1
  Re(fft(mascara * fft(x - recta), inverse = TRUE) / largo_serie) + recta
}

# Artista foco: el ilustrativo con mayor amplitud de trayectoria
artista_foco <- amplitud_series[
  artist_id %in% ids_ilustrativos & variable == VARIABLE_ILUSTRATIVA
][which.max(amplitud), artist_id]

serie_foco <- panel_long[.(artista_foco, VARIABLE_ILUSTRATIVA)]
serie_foco_pasa_bajos <- filtro_pasa_bajos_fft(serie_foco$valor_transformado)
serie_foco_pasa_bajos_sin_quitar_tendencia <- filtro_pasa_bajos_fft(serie_foco$valor_transformado, quitar_tendencia = FALSE)

In [ ]:
# ============================================================
# FIGURA 7: respuesta en frecuencia de las medias moviles, con el
# eje en periodo de la onda (meses)
# ============================================================

PERIODOS_EJE_RESPUESTA <- c(24, 12, 6, 4, 3, 2)

grafico_respuesta_medias_moviles <- function() {

  par(bg = COLOR_FONDO, mar = c(4.5, 5, 4, 1), las = 1, cex.axis = 0.85)

  plot(respuesta_filtros$ciclos_por_anio, respuesta_filtros$suavizado, type = "n", bty = "n",
       ylim = c(0, 1), xlab = "Período de la onda (meses)", ylab = "Proporción de la onda que pasa", xaxt = "n", yaxt = "n",
       main = "Qué ondas deja pasar cada media móvil", font.main = 1, cex.main = 1.25)
  # posiciones en ciclos por anio, etiquetas en meses (periodo = 12 / ciclos por anio)
  axis(1, at = 12 / PERIODOS_EJE_RESPUESTA, labels = PERIODOS_EJE_RESPUESTA, col = NA, col.ticks = "#BDBDBD")
  axis(2, at = seq(0, 1, 0.2), labels = format(seq(0, 1, 0.2), decimal.mark = ","), col = NA)
  abline(h = seq(0, 1, 0.2), col = COLOR_GRILLA)
  abline(h = 0.7, lty = 2, col = COLOR_REFERENCIA)
  lines(respuesta_filtros$ciclos_por_anio, respuesta_filtros$suavizado, col = COLOR_MEDIA_MOVIL, lwd = 4)
  lines(respuesta_filtros$ciclos_por_anio, respuesta_filtros$tendencia, col = COLOR_TENDENCIA, lwd = 4)

  # corte de cada filtro en 0,7, expresado como periodo
  frecuencias_corte <- unlist(cortes_media_movil[, .(corte_suavizado, corte_tendencia)])
  points(frecuencias_corte, c(0.7, 0.7), pch = 19, cex = 1.8, col = c(COLOR_MEDIA_MOVIL, COLOR_TENDENCIA))
  text(frecuencias_corte, c(0.7, 0.7) + 0.035, pos = 4, cex = 0.85, col = c(COLOR_MEDIA_MOVIL, COLOR_TENDENCIA),
       labels = paste0("corte: ", round(12 / frecuencias_corte), " meses"))

  legend("topright", bty = "n", cex = 0.95, lwd = 4, col = c(COLOR_MEDIA_MOVIL, COLOR_TENDENCIA),
         legend = c(paste0("media móvil ", VENTANAS_MEDIA_MOVIL[["suavizado"]], " meses"),
                    paste0("media móvil 2×", VENTANAS_MEDIA_MOVIL[["tendencia"]])))
}

guardar_figura("prepro_07_respuesta_medias_moviles.png", grafico_respuesta_medias_moviles, ancho = 1800, alto = 1200)


In [ ]:
# ============================================================
# FIGURA 8: pasa-bajos FFT sobre el artista foco, con y sin
# restar la tendencia lineal
# ============================================================

grafico_pasa_bajos_fft <- function(
  serie = serie_foco,
  pasa_bajos = serie_foco_pasa_bajos,
  pasa_bajos_sin_restar = serie_foco_pasa_bajos_sin_quitar_tendencia,
  colores = c(transformada = COLOR_SERIE_BASE, sin_restar = COLOR_ALERTA, restando = COLOR_RESULTADO)
) {

  par(bg = COLOR_FONDO, mar = c(3, 4.5, 4, 1), las = 1, cex.axis = 0.85)

  abrir_lienzo(range(c(serie$valor_transformado, pasa_bajos_sin_restar)),
               paste0("Pasa-bajos FFT (", BANDA_FILTRO_PASA_BAJOS, " armónicos): ", serie$artist_name[1], ", ",
                      en_minuscula_inicial(VARIABLES_PANEL[[VARIABLE_ILUSTRATIVA]])))
  lines(serie$month, serie$valor_transformado, col = colores[["transformada"]], lwd = 2.5)
  lines(serie$month, pasa_bajos_sin_restar, col = colores[["sin_restar"]], lwd = 3.5, lty = 2)
  lines(serie$month, pasa_bajos, col = colores[["restando"]], lwd = 4)
  legend("topleft", bty = "n", cex = 0.85, lwd = c(2.5, 3.5, 4), lty = c(1, 2, 1), seg.len = 2.5,
         col = colores[c("transformada", "sin_restar", "restando")],
         legend = c("transformada", "sin restar la tendencia", "restando la tendencia lineal"))
}

guardar_figura("prepro_08_pasa_bajos_fft.png", grafico_pasa_bajos_fft, ancho = 2200, alto = 1100)


In [ ]:
# ============================================================
# FIGURA 8, VERSION EN TONOS DEL ARTISTA: el pasa-bajos FFT sobre
# el artista de la version en tonos de la figura 5: transformada
# en su color propio, alternativa sin restar la tendencia en el
# gris claro de la figura 5 y resultado en el tono oscuro
# ============================================================

serie_version_tonos <- panel_long[.(ARTISTA_VERSION_TONOS, VARIABLE_ILUSTRATIVA)]

guardar_figura(
  paste0("prepro_08_pasa_bajos_fft_", SUFIJO_VERSION_TONOS, ".png"),
  function() grafico_pasa_bajos_fft(
    serie = serie_version_tonos,
    pasa_bajos = filtro_pasa_bajos_fft(serie_version_tonos$valor_transformado),
    pasa_bajos_sin_restar = filtro_pasa_bajos_fft(serie_version_tonos$valor_transformado, quitar_tendencia = FALSE),
    colores = c(transformada = TONOS_VERSION_ARTISTA[["propio"]],
                sin_restar = COLOR_MEDIA_MOVIL_VERSION_TONOS,
                restando = TONOS_VERSION_ARTISTA[["oscuro"]])
  ),
  ancho = 2200, alto = 1100
)


La media de 3 meses conserva las oscilaciones de más de medio año; la 2x12 elimina la estacionalidad anual. Sin restar la tendencia, el pasa-bajos FFT deforma los extremos.

## 9. Crecimiento mensual

Derivada FIR centrada, `filter(x, c(0.5, 0, -0.5))`, sobre la serie suavizada: con `log1p` aproxima la tasa de crecimiento mensual. Es el insumo de la segmentación.

In [ ]:
# ============================================================
# DERIVADA FIR: CRECIMIENTO MENSUAL
# ============================================================

derivada_fir_centrada <- function(x) {
  largo_serie <- length(x)
  derivada <- as.numeric(stats::filter(x, c(0.5, 0, -0.5), sides = 2))
  derivada[1] <- x[2] - x[1]
  derivada[largo_serie] <- x[largo_serie] - x[largo_serie - 1]
  derivada
}

panel_long[
  ,
  crecimiento_mensual := derivada_fir_centrada(valor_suavizado),
  by = .(artist_id, variable)
]

In [ ]:
# ------------------------------------------------------------
# CONTROL: crecimiento mensual tipico por variable, y el mes de
# mayor crecimiento de los artistas de la figura de presentacion
# ------------------------------------------------------------

panel_long[
  ,
  .(mediana_pct = round(100 * median(crecimiento_mensual), 2),
    p95_pct = round(100 * quantile(crecimiento_mensual, 0.95), 2)),
  by = variable
]

panel_long[
  artist_id %in% ARTISTAS_FIGURA_PRESENTACION,
  .SD[which.max(crecimiento_mensual), .(month, crecimiento_pct = round(100 * crecimiento_mensual, 1))],
  by = .(artist_name, variable)
]

## 10. Features por artista

Rasgos de cada serie para **describir** los clusters (no son la entrada del clustering). Cada uno se calcula con una técnica de la materia:

| Feature | Técnica | Clase |
|---|---|---|
| `nivel_log_medio` | media de `log1p` (tamaño del artista) | 01 |
| `pendiente_anual` | regresión lineal sobre el tiempo | 10 |
| `crecimiento_total` | diferencia de `log1p` suavizado entre el último y el primer mes | 01 y 10 |
| `volatilidad_residuo` | desvío de la serie menos la tendencia 2x12 | 03 |
| `crecimiento_max`, `mes_crecimiento_max` (fecha), `caida_max` | derivada FIR, sin los 2 meses de cada borde (el inicio es basal) | 08 |
| `aceleracion` | pendiente anual del crecimiento mensual (negativa = desacelera) | 08 y 10 |
| `picos_aislados`, `pct_imputado` | limpieza e interpolación | 06 |
| `energia_lenta`, `energia_rapida` | FFT sin tendencia: energía en oscilaciones de 1 año o más y de menos de 6 meses | 02 y 04 |

In [ ]:
# ============================================================
# FEATURES POR ARTISTA Y VARIABLE
# Tabla ancha: una fila por artista, una columna por feature y
# variable (feature__variable).
# ============================================================

features_serie <- panel_long[
  ,
  {
  interior <- mes_indice > MESES_BORDE_SIN_EXTREMOS & mes_indice <= N_MESES_PANEL - MESES_BORDE_SIN_EXTREMOS
  .(
    nivel_log_medio = mean(log_valor),
    pendiente_anual = 12 * unname(coef(lm(valor_suavizado ~ mes_indice))[2]),
    crecimiento_total = valor_suavizado[.N] - valor_suavizado[1],
    volatilidad_residuo = sd(valor_transformado - tendencia),
    crecimiento_max = max(crecimiento_mensual[interior]),
    mes_crecimiento_max = month[interior][which.max(crecimiento_mensual[interior])],
    caida_max = min(crecimiento_mensual[interior]),
    aceleracion = 12 * unname(coef(lm(crecimiento_mensual[interior] ~ mes_indice[interior]))[2]),
    picos_aislados = as.numeric(sum(pico_aislado)),
    pct_imputado = 100 * mean(imputado)
  )
  },
  by = .(artist_id, variable)
]

# Energia por banda, con el mismo criterio que la tabla del bloque 8
energia_serie <- espectro_panel[
  artist_id %in% artistas_panel$artist_id,
  .(
    energia_lenta = sum(amplitud_transformada[ciclos_por_anio <= 1]^2) / sum(amplitud_transformada^2),
    energia_rapida = sum(amplitud_transformada[ciclos_por_anio > 2]^2) / sum(amplitud_transformada^2)
  ),
  by = .(artist_id, variable)
]

features_serie <- merge(features_serie, energia_serie, by = c("artist_id", "variable"))

COLUMNAS_FEATURES <- setdiff(names(features_serie), c("artist_id", "variable"))

features_artista <- dcast(
  features_serie,
  artist_id ~ variable,
  value.var = COLUMNAS_FEATURES,
  sep = "__"
)

features_artista <- merge(artistas_panel, features_artista, by = "artist_id")

In [ ]:
# ------------------------------------------------------------
# SANITY CHECK DE LAS FEATURES: una fila por artista del panel,
# sin NA, mes de crecimiento maximo mas frecuente y mediana de
# cada feature numerica por variable
# ------------------------------------------------------------

stopifnot(
  "La tabla no tiene una fila por artista del panel" =
    nrow(features_artista) == nrow(artistas_panel) && !anyDuplicated(features_artista$artist_id),
  "Hay features con NA" =
    !anyNA(features_artista[, !"artist_name"])
)

cat("Artistas:", nrow(features_artista), "| columnas de features:",
    ncol(features_artista) - ncol(artistas_panel), "\n\n")

# Mes del crecimiento maximo: el mas frecuente por variable
print(features_serie[, .N, by = .(variable, mes_crecimiento_max)][order(-N)][, .SD[1], by = variable])

dcast(
  melt(features_serie[, !"mes_crecimiento_max"], id.vars = c("artist_id", "variable"), variable.name = "feature")[
    ,
    .(mediana = signif(median(value), 3)),
    by = .(feature, variable)
  ],
  feature ~ variable,
  value.var = "mediana"
)

In [ ]:
# ============================================================
# FIGURA: la desaceleración. Crecimiento mensual típico (mediana
# entre artistas) mes a mes, sin los meses de borde.
# Arriba las tres audiencias (el mensaje); abajo los conteos en
# playlists, más ruidosos, con escala propia. Escala de grises y
# nombre de cada serie al final de la curva, con el porcentaje de
# series que desaceleran.
# ============================================================

desaceleracion_mensual <- panel_long[
  mes_indice > MESES_BORDE_SIN_EXTREMOS & mes_indice <= N_MESES_PANEL - MESES_BORDE_SIN_EXTREMOS,
  .(crecimiento_mediano_pct = 100 * median(crecimiento_mensual)),
  by = .(variable, month)
]

pct_desacelera <- features_serie[, .(pct_desacelera = round(100 * mean(aceleracion < 0))), by = variable]

NOMBRE_CORTO_VARIABLE <- c(
  spotify_followers = "Spotify", instagram_followers = "Instagram", youtube_channel_subscribers = "YouTube",
  spotify_playlist_count = "Playlists Spotify", youtube_playlist_count = "Playlists YouTube"
)
GRIS_VARIABLE <- c(
  spotify_followers = "#111111", instagram_followers = "#6B6B6B", youtube_channel_subscribers = "#A6A6A6",
  spotify_playlist_count = "#4A4A4A", youtube_playlist_count = "#A6A6A6"
)

desaceleracion_mensual <- merge(desaceleracion_mensual, pct_desacelera, by = "variable")
desaceleracion_mensual[, `:=`(
  grupo = factor(fifelse(grepl("playlist", variable), "Conteos en playlists", "Audiencias"),
                 levels = c("Audiencias", "Conteos en playlists")),
  etiqueta = paste0(NOMBRE_CORTO_VARIABLE[variable], "
", pct_desacelera, "% desacelera")
)]
etiquetas_final <- desaceleracion_mensual[, .SD[which.max(month)], by = variable]

# Sanity check: cinco series con su porcentaje
stopifnot(nrow(etiquetas_final) == 5, !anyNA(etiquetas_final$pct_desacelera))

grafico_desaceleracion <- ggplot(desaceleracion_mensual,
                                 aes(x = month, y = crecimiento_mediano_pct, colour = variable)) +
  geom_hline(yintercept = 0, colour = "#BDBDBD", linewidth = 0.4) +
  geom_line(linewidth = 1.3) +
  geom_text_repel(data = etiquetas_final, aes(label = etiqueta), hjust = 0, nudge_x = 40, direction = "y",
                  family = "LWR Mono", fontface = "bold", size = 3.6, lineheight = 0.95,
                  segment.colour = "#BDBDBD", min.segment.length = 0, box.padding = 0.35) +
  facet_wrap(~ grupo, ncol = 1, scales = "free_y") +
  scale_colour_manual(values = GRIS_VARIABLE, guide = "none") +
  scale_x_date(breaks = as.Date(paste0(2023:2026, "-01-01")), date_labels = "%Y", expand = expansion(mult = c(0.02, 0.3))) +
  labs(
    title = "Crecen siempre, pero cada vez más despacio",
    subtitle = paste0(
      "Crecimiento mensual del artista típico (mediana entre artistas).\n",
      "Porcentaje: series cuyo ritmo de crecimiento baja en el período"
    ),
    x = NULL, y = "Crecimiento mensual (%)"
  ) +
  theme_minimal(base_size = 13, base_family = "LWR Mono") +
  theme(
    plot.title = element_text(family = "LWR Titulo", size = 26, colour = "#111111", margin = margin(b = 8)),
    plot.subtitle = element_text(size = 12, colour = "#4A4A4A", lineheight = 1.15, margin = margin(b = 14)),
    strip.text = element_text(face = "bold", size = 13, colour = "#111111", hjust = 0),
    axis.title = element_text(colour = "#4A4A4A", size = 12),
    axis.text = element_text(colour = "#6B6B6B", size = 10),
    panel.grid.minor = element_blank(),
    panel.grid.major = element_line(colour = "#EDEDED"),
    panel.spacing = unit(18, "pt"),
    plot.title.position = "plot",
    plot.background = element_rect(fill = "white", colour = NA),
    plot.margin = margin(18, 18, 12, 18)
  )

ggsave(file.path(FIGURES_DIR, "prepro_09_desaceleracion.png"), grafico_desaceleracion,
       width = 11, height = 7.5, dpi = 300, bg = "white", device = ragg::agg_png)

# El dispositivo del notebook no conoce las tipografías registradas:
# se muestra el PNG guardado
IRdisplay::display_png(file = file.path(FIGURES_DIR, "prepro_09_desaceleracion.png"))

pct_desacelera


## 11. Exportación

El panel preprocesado conserva la serie cruda y las marcas de cada etapa, para rastrear cualquier resultado hasta el dato original.

In [ ]:
# ============================================================
# EXPORTACION DEL PANEL PREPROCESADO Y DE LAS FEATURES
# ============================================================

fwrite(
  panel_long[
    ,
    .(artist_id, artist_name, cm_artist_rank, decil_marco, variable, month,
      valor_crudo, interpolated, pico_aislado, imputado, empalme, log_valor,
      valor_transformado, valor_suavizado, tendencia, crecimiento_mensual)
  ],
  ARCHIVO_PANEL_PREPROCESADO
)

ARCHIVO_FEATURES_ARTISTA <- sub("_long[.]csv$", "_features_artista.csv", ARCHIVO_PANEL_PREPROCESADO)

fwrite(features_artista, ARCHIVO_FEATURES_ARTISTA)

cat("Exportado:", ARCHIVO_PANEL_PREPROCESADO, "\n",
    "Filas:", nrow(panel_long), "\n",
    "Features:", ARCHIVO_FEATURES_ARTISTA, "-", nrow(features_artista), "artistas x",
    ncol(features_artista), "columnas\n")